# 5. Standardize Extracted Data-Source Names

This notebook starts from the compiled LLM corpus results and creates machine standardization proposals plus the workbook for human review. The manually cleaned workbook is applied later in notebook `6`, which creates the finalized standardized long source table for downstream bibliometric and network analysis.

All paths are resolved through `project_config.py`. Set `DATA_INTERDEPENDENCIES_DATA_ROOT` before running the notebook.


## 1. Set Local Input And Output Paths


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

from pathlib import Path

# Local testing root. Later, replace this single value with the envpol share-drive root.
# DATA_ROOT is configured through project_config.
CODE_ROOT = REPO_ROOT
TEXT_MINING_CODE_ROOT = CODE_ROOT / "text_mining"

COMPILED_DIR = DATA_ROOT / "llm_labelled_data" / "compiled"
PROCESSED_DIR = DATA_ROOT / "processed_data"

INPUT_PATH = COMPILED_DIR
OUTPUT_PATH = COMPILED_DIR

LLM_MASTER_FILE = COMPILED_DIR / "llm_labelled_results_master.csv"
LLM_LONG_FILE = COMPILED_DIR / "llm_labelled_data_sources_long.csv"
REVIEW_FILE = COMPILED_DIR / "llm_labelled_datasource_review.csv"
REVIEW_EXCEL_FILE = COMPILED_DIR / "llm_labelled_datasource_review.xlsx"
MANUAL_REVIEW_FILE = COMPILED_DIR / "manually_cleaned" / "llm_labelled_datasource_review.xlsx"
CURATED_ALIAS_FILE = COMPILED_DIR / "llm_labelled_datasource_curated_aliases.csv"

# Backwards-compatible aliases for older cells in this notebook.
ROOT = DATA_ROOT
input_path = INPUT_PATH

print(f"Data root: {DATA_ROOT}")
print(f"Compiled LLM folder: {COMPILED_DIR}")
print(f"Manual review file expected after human cleaning: {MANUAL_REVIEW_FILE}")


## 2. Extract raw names and build cleaned-name helpers

This section reads the current master file, keeps successful LLM-labelled rows, parses the raw data-source mentions, and builds cleaned-name helper tables. The review workflow itself starts in the next section, where one ordered rule engine creates the machine proposal for each raw name.


In [ ]:
import json
import re

import pandas as pd


BRACKETED_NUMERIC_SOURCE_RE = re.compile(
    r"^(?:\[\s*-?\d+(?:\s*[,;/\-]\s*-?\d+)*\s*\]\s*)+$"
)


def parse_json_like_source_text(text):
    candidates = [text]
    overescaped_quote = "\\\\" + chr(34)
    escaped_quote = "\\" + chr(34)
    if overescaped_quote in text:
        candidates.append(text.replace(overescaped_quote, escaped_quote))

    for candidate in candidates:
        try:
            return json.loads(candidate)
        except (json.JSONDecodeError, TypeError):
            continue
    return None


def numeric_json_list_to_reference_marker(items):
    if not items or not all(isinstance(item, (int, float)) and not isinstance(item, bool) for item in items):
        return None
    labels = []
    for item in items:
        labels.append(str(int(item)) if isinstance(item, float) and item.is_integer() else str(item))
    return "[" + ",".join(labels) + "]"


COMMA_SOURCE_LIST_PART_RE = re.compile(r"^[A-Za-z0-9][A-Za-z0-9 .&'’/()\-]{1,80}$")
YEAR_ONLY_PART_RE = re.compile(r"^(?:19|20)\d{2}[a-z]?$", flags=re.IGNORECASE)


def split_delimited_source_text(text):
    semi_parts = [part.strip() for part in str(text or "").split(";") if part.strip()]
    if len(semi_parts) != 1:
        return semi_parts

    only_part = semi_parts[0]
    comma_parts = [part.strip() for part in only_part.split(",") if part.strip()]
    if len(comma_parts) < 3:
        return semi_parts
    if any(YEAR_ONLY_PART_RE.match(part) for part in comma_parts):
        return semi_parts
    plausible_parts = [part for part in comma_parts if COMMA_SOURCE_LIST_PART_RE.match(part)]
    short_or_acronym_parts = [
        part for part in comma_parts
        if re.fullmatch(r"[A-Z][A-Z0-9.&/ -]{1,25}", part) or len(part.split()) <= 5
    ]
    if len(plausible_parts) == len(comma_parts) and len(short_or_acronym_parts) >= 2:
        return comma_parts
    return semi_parts


def extract_source_strings(parsed):
    if isinstance(parsed, list):
        numeric_marker = numeric_json_list_to_reference_marker(parsed)
        if numeric_marker:
            return [numeric_marker]

        cleaned_items = []
        for item in parsed:
            cleaned_items.extend(extract_source_strings(item))
        return cleaned_items

    if isinstance(parsed, dict):
        item = parsed.get("name") or parsed.get("data_source") or parsed.get("source")
        return extract_source_strings(item) if item is not None else []

    if isinstance(parsed, str):
        text = parsed.strip()
        if not text:
            return []
        if BRACKETED_NUMERIC_SOURCE_RE.match(text):
            return [text]
        nested = parse_json_like_source_text(text)
        if isinstance(nested, list):
            return extract_source_strings(nested)
        return split_delimited_source_text(text)

    text = str(parsed).strip()
    return [text] if text else []


def parse_data_sources(value):
    if pd.isna(value):
        return []

    text = str(value).strip()
    if not text:
        return []

    parsed = parse_json_like_source_text(text)
    if parsed is None:
        parsed = text

    seen = []
    for item in extract_source_strings(parsed):
        if item and item not in seen:
            seen.append(item)
    return seen


def join_data_sources(value):
    return "; ".join(parse_data_sources(value))


def searchable_doi_from_file_name(file_name):
    stem = Path(str(file_name or "")).name
    stem = re.sub(r"\.txt$", "", stem, flags=re.IGNORECASE).strip()
    match = re.match(r"^(10\.\d{4,9})[_/](.+)$", stem, flags=re.IGNORECASE)
    if not match:
        return ""
    suffix = match.group(2).replace("_", "/")
    return f"https://doi.org/{match.group(1)}/{suffix}"


def split_database_names(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(";") if part and part.strip()]


COMPARISON_SUFFIX = re.compile(r"\s*\(comparison\)\s*$", flags=re.IGNORECASE)
TRAILING_PUNCTUATION = re.compile(r"[\s,;:.]+$")
WHITESPACE = re.compile(r"\s+")


def cleanup_database_name(name):
    cleaned = str(name).replace("’", "'").strip()
    cleaned = COMPARISON_SUFFIX.sub("", cleaned)
    cleaned = TRAILING_PUNCTUATION.sub("", cleaned)
    cleaned = WHITESPACE.sub(" ", cleaned)
    return cleaned


df = pd.read_csv(LLM_MASTER_FILE, low_memory=False)
required_columns = {"db_use", "data_sources_json"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"LLM master is missing required columns: {sorted(missing_columns)}")

if "is_success" in df.columns:
    is_success = df["is_success"]
    if is_success.dtype == object:
        is_success = is_success.astype("string").str.lower().isin({"true", "1", "yes"})
    df = df.loc[is_success.fillna(False)].copy()

if "file_name" in df.columns:
    df["doi"] = df["file_name"].map(searchable_doi_from_file_name)

TEXT_IDENTIFIER_COLUMNS = [
    column
    for column in ["global_index", "file_name", "doi", "relative_path", "source_subfolder"]
    if column in df.columns
]

df["databases"] = df["data_sources_json"].map(join_data_sources)
db_use_numeric = pd.to_numeric(df["db_use"], errors="coerce")
database_rows = df.loc[db_use_numeric.eq(1)].copy()

print(f"Loaded LLM master: {LLM_MASTER_FILE}")
print(f"Successful LLM-labelled rows used: {len(df):,}")
print(f"Rows marked as data-source use: {len(database_rows):,}")


## 3. Create machine proposals for review

This step creates one row per raw extracted name, applies the ordered rule engine, and writes one automated proposal file. The reviewer-facing columns are created blank and are not read back into this notebook. Some rows, such as `citation_or_reference`, can remain automated suggestions without being manually reviewed.

The curated alias file `llm_labelled_datasource_curated_aliases.csv` is an editable rule table for GBIF-like cases. Add rows there when you find another known resource/provider whose spelling variants, expanded names, domains, or approved family prefixes should collapse to one canonical name.


In [ ]:
# Machine proposal export layer
#
# This cell exports full-dataset CSV and Excel files with automated suggestions.
# Human-review columns are intentionally left blank.

import csv
import time

REVIEW_FILE = OUTPUT_PATH / "llm_labelled_datasource_review.csv"
REVIEW_EXCEL_FILE = OUTPUT_PATH / "llm_labelled_datasource_review.xlsx"

PROGRESS_EVERY = 1000

CSV_WRITE_KWARGS = {
    "index": False,
    "encoding": "utf-8-sig",
    "quoting": csv.QUOTE_MINIMAL,
    "escapechar": "\\",
    "doublequote": True,
    "lineterminator": "\n",
}


def write_pipeline_csv(frame, path):
    frame.to_csv(path, **CSV_WRITE_KWARGS)


def write_pipeline_excel(frame, path):
    frame.to_excel(path, index=False, sheet_name="review")


def compact_unique_join(values, max_items=3):
    seen = []
    for value in values:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text and text not in seen:
            seen.append(text)
        if len(seen) >= max_items:
            break
    return " | ".join(seen)


def review_lookup_key(value):
    text = cleanup_database_name(value)
    text = text.replace("&", " and ").casefold()
    text = re.sub(r"(?i)['`]s\b", "", text)
    text = re.sub(r"['`]+", "", text)
    text = re.sub(
        r"\b(database|data base|dataset|datasets|data set|data portal|portal|web service|website|web site|archive|repository)\b\s*$",
        "",
        text,
    )
    text = re.sub(r"[^a-z0-9+]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def raw_review_lookup_key(value):
    text = cleanup_database_name(value)
    text = text.replace("&", " and ").casefold()
    text = re.sub(r"(?i)['`]s\b", "", text)
    text = re.sub(r"['`]+", "", text)
    text = re.sub(r"[^a-z0-9+]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


SOFTWARE_EXACT_KEYS = {
    "aries",
    "r",
    "arcgis",
    "arcmap",
    "matlab",
    "imagej",
    "genalex",
    "mega",
    "mega x",
    "clustal omega",
    "spss",
    "ibm spss",
    "sas",
    "stata",
    "past",
    "pc ord",
    "winbugs",
    "maxent",
    "blast",
    "mega",
    "beast",
    "structure",
    "mrbayes",
    "figtree",
    "clustalw",
    "clustal x",
    "trimmomatic",
    "mothur",
    "qiime",
    "admixture",
    "excel",
    "microsoft excel",
    "vegan",
    "lme4",
    "qgis",
    "co2sys",
    "busco",
    "blastn",
    "blastp",
    "blastx",
    "tblastn",
    "ncbi blast",
    "structure harvester",
    "estimates",
    "hysplit",
    "raxml",
    "bwa",
    "bowtie",
    "star",
    "trinity",
    "muscle",
    "mafft",
    "en mtools",
    "enmtools",
    "mark",
    "program mark",
    "rast",
    "rast server",
    "smart",
    "picrust",
    "genepop",
    "megan",
    "arlequin",
    "diva gis",
    "diva-gis",
    "geneious",
}

BIBLIOGRAPHIC_EXACT_KEYS = {
    "web of science",
    "scopus",
    "google scholar",
    "pubmed",
    "science citation index",
    "isi web of knowledge",
    "isi web of science",
    "science direct",
    "sciencedirect",
    "biosis",
    "biosis biological abstracts",
    "biosis citation index",
    "biosis citations index",
    "cab abstracts",
    "cabi cab abstracts",
    "zoological record",
    "dimensions",
}

HOLDER_EXACT_KEYS = {
    "usgs",
    "united states geological survey",
    "us geological survey",
    "u s geological survey",
    "noaa",
    "nasa",
    "fao",
    "ecmwf",
    "esa",
    "world bank",
    "iucn",
    "google",
    "ibge",
    "inmet",
    "usda",
    "who",
    "environment canada",
    "china meteorological data sharing service system",
    "cru",
    "climate research unit",
    "climatic research unit",
    "dwd",
    "deutscher wetterdienst",
    "german weather service",
    "german meteorological service",
    "noaa ncdc",
    "ncdc",
    "noaa ncei",
    "ncei",
    "noaa cpc",
    "cpc",
    "noaa esrl",
    "esrl",
    "us census bureau",
    "us census",
    "china meteorological administration",
    "cma",
    "resdc",
    "data center for resources and environmental sciences chinese academy of sciences",
}

SUGGESTED_CANONICAL_MAP = {
    "genbank": "GenBank",
    "gene bank": "GenBank",
    "genebank": "GenBank",
    "genbank nucleotide": "GenBank",
    "ncbi genbank": "GenBank",
    "ncbi nucleotide": "NCBI Nucleotide",
    "ncbi nucleotide collection nr nt": "NCBI Nucleotide",
    "ncbi": "NCBI",
    "national center for biotechnology information ncbi": "NCBI",
    "national centre for biotechnology information ncbi": "NCBI",
    "sra": "NCBI SRA",
    "sequence read archive": "NCBI SRA",
    "sequence read archive sra": "NCBI SRA",
    "ncbi sra": "NCBI SRA",
    "ncbi sequence read archive": "NCBI SRA",
    "ncbi sequence read archive sra": "NCBI SRA",
    "short read archive": "NCBI SRA",
    "short read archive sra": "NCBI SRA",
    "geo": "NCBI GEO",
    "gene expression omnibus": "NCBI GEO",
    "gene expression omnibus geo": "NCBI GEO",
    "refseq": "NCBI RefSeq",
    "ncbi refseq": "NCBI RefSeq",
    "ncbi biosample": "NCBI BioSample",
    "biosample": "NCBI BioSample",
    "ncbi bioproject": "NCBI BioProject",
    "bioproject": "NCBI BioProject",
    "dbgap": "dbGaP",
    "ncbi dbgap": "dbGaP",
    "ncbi taxonomy": "NCBI Taxonomy",
    "ncbi taxonomy database": "NCBI Taxonomy",
    "ncbi taxonomy browser": "NCBI Taxonomy",
    "ncbi genome": "NCBI Genome",
    "ncbi genome database": "NCBI Genome",
    "ncbi nonredundant": "NCBI nr",
    "ncbi nonredundant database": "NCBI nr",
    "ncbi nonredundant protein database": "NCBI nr",
    "ncbi nonredundant nucleotide database": "NCBI nr",
    "ncbi nr": "NCBI nr",
    "nr database": "NCBI nr",
    "cru": "CRU",
    "climate research unit": "CRU",
    "climatic research unit": "CRU",
    "cru ts": "CRU TS",
    "cru ts3": "CRU TS",
    "world ocean database": "World Ocean Database",
    "world ocean database wod": "World Ocean Database",
    "esa cci": "ESA CCI",
    "esa cci land cover": "ESA CCI",
    "esa cci soil moisture": "ESA CCI",
    "esa cci sm": "ESA CCI",
    "forest inventory and analysis": "Forest Inventory and Analysis",
    "forest inventory analysis": "Forest Inventory and Analysis",
    "fia": "Forest Inventory and Analysis",
    "tiger line": "TIGER/Line Shapefiles",
    "tiger line shapefile": "TIGER/Line Shapefiles",
    "tiger line shapefiles": "TIGER/Line Shapefiles",
    "ucsc genome browser": "UCSC Genome Browser",
    "ucsc genome": "UCSC Genome Browser",
    "whole genome shotgun": "NCBI WGS",
    "wgs": "NCBI WGS",
    "arrayexpress": "ArrayExpress",
    "dwd": "DWD",
    "deutscher wetterdienst": "DWD",
    "german weather service": "DWD",
    "german meteorological service": "DWD",
    "noaa ncdc": "NOAA NCDC",
    "ncdc": "NOAA NCDC",
    "national climatic data center": "NOAA NCDC",
    "national climate data center": "NOAA NCDC",
    "noaa ncei": "NOAA NCEI",
    "ncei": "NOAA NCEI",
    "national centers for environmental information": "NOAA NCEI",
    "noaa cpc": "NOAA CPC",
    "cpc": "NOAA CPC",
    "climate prediction center": "NOAA CPC",
    "noaa esrl": "NOAA ESRL",
    "esrl": "NOAA ESRL",
    "us census bureau": "US Census Bureau",
    "us census": "US Census Bureau",
    "china meteorological administration": "CMA",
    "cma": "CMA",
    "cluster of orthologous groups": "COG",
    "clusters of orthologous groups": "COG",
    "global runoff data centre": "GRDC",
    "global runoff data center": "GRDC",
    "germplasm resources information network": "GRIN",
    "grin": "GRIN",
    "kegg": "KEGG",
    "pfam": "Pfam",
    "uniprot": "UniProt",
    "uniprotkb": "UniProt",
    "silva": "SILVA",
    "greengenes": "Greengenes",
    "rdp": "RDP",
    "bold": "BOLD",
    "barcoding of life data system bold": "BOLD",
    "gbif": "GBIF",
    "global biodiversity information facility gbif": "GBIF",
    "iucn red list": "IUCN Red List",
    "iucn red list of threatened species": "IUCN Red List",
    "worldclim": "WorldClim",
    "world ocean atlas": "World Ocean Atlas",
    "woa": "World Ocean Atlas",
    "woa09": "World Ocean Atlas",
    "woa98": "World Ocean Atlas",
    "era interim": "ERA-Interim",
    "era interim reanalysis": "ERA-Interim",
    "era5": "ERA5",
    "ncep": "NCEP Reanalysis",
    "ncep ncar reanalysis": "NCEP/NCAR Reanalysis",
    "cru": "CRU",
    "cru ts": "CRU TS",
    "hadcrut": "HadCRUT",
    "cmip5": "CMIP5",
    "cmip6": "CMIP6",
    "modis": "MODIS",
    "moderate resolution imaging spectroradiometer modis": "MODIS",
    "landsat": "Landsat",
    "landsat etm+": "Landsat ETM+",
    "landsat tm": "Landsat TM",
    "sea wifs": "SeaWiFS",
    "seawifs": "SeaWiFS",
    "trmm": "TRMM",
    "gpcp": "GPCP",
    "grace": "GRACE",
    "srtm": "SRTM",
    "aster": "ASTER",
    "usgs": "USGS",
    "united states geological survey": "USGS",
    "us geological survey": "USGS",
    "u s geological survey": "USGS",
    "usgs earth explorer": "USGS EarthExplorer",
    "earth explorer": "USGS EarthExplorer",
    "google earth": "Google Earth",
    "google earth pro": "Google Earth",
    "google maps": "Google Maps",
    "google map": "Google Maps",
    "web of science": "Web of Science",
    "scopus": "Scopus",
    "google scholar": "Google Scholar",
    "pubmed": "PubMed",
    "biosis": "BIOSIS",
    "biosis biological abstracts": "BIOSIS Biological Abstracts",
    "biosis citation index": "BIOSIS Citation Index",
    "biosis citations index": "BIOSIS Citation Index",
    "cab abstracts": "CAB Abstracts",
    "cabi cab abstracts": "CAB Abstracts",
    "zoological record": "Zoological Record",
    "dimensions": "Dimensions",
    "dryad": "Dryad",
    "figshare": "Figshare",
    "pangaea": "PANGAEA",
    "fishbase": "FishBase",
    "treebase": "TreeBASE",
    "mycobank": "MycoBank",

    "cog": "COG",
    "clusters of orthologous groups cog": "COG",
    "unite": "UNITE",
    "mg rast": "MG-RAST",
    "mg-rast": "MG-RAST",
    "european nucleotide archive": "European Nucleotide Archive",
    "european nucleotide archive ena": "European Nucleotide Archive",
    "ena": "European Nucleotide Archive",
    "embl": "EMBL Nucleotide Sequence Database",
    "embl nucleotide sequence database": "EMBL Nucleotide Sequence Database",
    "ddbj": "DDBJ",
    "dna data bank of japan": "DDBJ",
    "ddbj embl genbank": "INSDC",
    "insdc": "INSDC",
    "ribosomal database project": "RDP",
    "ribosomal database project rdp": "RDP",
    "rdp classifier": "RDP Classifier",
    "swiss prot": "Swiss-Prot",
    "swissprot": "Swiss-Prot",
    "uniprotkb swiss prot": "Swiss-Prot",
    "pdb": "Protein Data Bank",
    "protein data bank": "Protein Data Bank",
    "rcsb pdb": "Protein Data Bank",
    "go": "Gene Ontology",
    "gene ontology": "Gene Ontology",
    "global biodiversity information facility": "GBIF",
    "dryad digital repository": "Dryad",
    "dryad digital": "Dryad",
    "figshare repository": "Figshare",
    "zenodo": "Zenodo",
    "worms": "WoRMS",
    "world register of marine species": "WoRMS",
    "world register of marine species worms": "WoRMS",
    "shuttle radar topography mission srtm": "SRTM",
    "aster gdem": "ASTER GDEM",
    "avhrr": "AVHRR",
    "advanced very high resolution radiometer avhrr": "AVHRR",
    "sentinel 1": "ESA",
    "sentinel 2": "ESA",
    "sentinel-1": "ESA",
    "sentinel-2": "ESA",
    "hadisst": "HadISST",
    "era 40": "ERA-40",
    "era-40": "ERA-40",
    "prism": "PRISM",
    "aeronet": "AERONET",
    "aviso": "AVISO",
    "gebco": "GEBCO",
    "openstreetmap": "OpenStreetMap",
    "open street map": "OpenStreetMap",
    "natural earth": "Natural Earth",
    "nsidc": "NSIDC",
    "argo": "Argo",
    "fluxnet": "FLUXNET",
    "fluxnet2015": "FLUXNET2015",
    "cazy": "CAZy",
    "carbohydrate active enzymes database cazy": "CAZy",
    "interpro": "InterPro",
    "phytozome": "Phytozome",
    "pride": "PRIDE",
    "david": "DAVID",
    "the plant list": "The Plant List",
    "world bank": "World Bank",
    "world bank open data": "World Bank Open Data",
    "world development indicators": "World Development Indicators",
    "wdi": "World Development Indicators",
    "worldwide governance indicators": "Worldwide Governance Indicators",
    "world governance indicators": "Worldwide Governance Indicators",
    "wgi": "Worldwide Governance Indicators",
    "faostat": "FAOSTAT",
    "global administrative unit layers": "FAO GAUL",
    "global administrative unit layers gaul": "FAO GAUL",
    "gaul": "FAO GAUL",
    "global forest resources assessment": "FAO FRA",
    "forest resources assessment": "FAO FRA",
    "fao fra": "FAO FRA",
    "ncep doe reanalysis 2 r 2": "NCEP/DOE Reanalysis 2",
    "ncep doe reanalysis 2": "NCEP/DOE Reanalysis 2",
    "ncep cfsr": "NCEP CFSR",
    "climate forecast system reanalysis": "NCEP CFSR",
    "ncep sst": "NCEP SST",
    "noaa oi sst v2": "NOAA OI SST V2",
    "noaa oisst": "NOAA OISST",
    "oisst": "NOAA OISST",
    "noaa ersst": "NOAA ERSST",
    "ersst": "NOAA ERSST",
    "noaa coral reef watch": "NOAA Coral Reef Watch",
    "noaa coastwatch oceanwatch": "NOAA CoastWatch/OceanWatch",
    "noaa national weather service": "NOAA National Weather Service",
    "national weather service": "NOAA National Weather Service",
    "noaa office for coastal management": "NOAA Office for Coastal Management",
    "noaa nos": "NOAA NOS",
    "nos": "NOAA NOS",
    "nasa power": "NASA POWER",
    "nasa-power": "NASA POWER",
    "lp daac": "NASA LP DAAC",
    "lpdaac": "NASA LP DAAC",
    "nasa lp daac": "NASA LP DAAC",
    "nasa lpdaac": "NASA LP DAAC",
    "nasa obpg": "NASA OBPG",
    "obpg": "NASA OBPG",
    "ocean biology processing group": "NASA OBPG",
    "nasa measures": "NASA MEaSUREs",
    "usgs breeding bird survey": "USGS Breeding Bird Survey",
    "north american breeding bird survey": "USGS Breeding Bird Survey",
    "usgs nawqa": "USGS NAWQA",
    "nawqa": "USGS NAWQA",
    "usgs nas": "USGS NAS",
    "national land cover database": "National Land Cover Database",
    "nlcd": "National Land Cover Database",
    "ecmwf oras5": "ECMWF ORAS5",
    "oras5": "ECMWF ORAS5",
    "ecmwf ifs": "ECMWF IFS",
    "gimms": "GIMMS",
    "noaa": "NOAA",
    "national oceanic and atmospheric administration": "NOAA",
    "nasa": "NASA",
    "national aeronautics and space administration": "NASA",
    "fao": "FAO",
    "food and agriculture organization": "FAO",
    "ecmwf": "ECMWF",
    "esa": "ESA",
    "european centre for medium range weather forecasts": "ECMWF",
    "european center for medium range weather forecasts": "ECMWF",
    "iucn": "IUCN",
    "international union for conservation of nature": "IUCN",
    "ibge": "IBGE",
    "inmet": "INMET",
    "ipcc": "IPCC",
    "nist": "NIST",
    "atcc": "ATCC",
    "science direct": "ScienceDirect",
    "sciencedirect": "ScienceDirect",
    "isi web of science": "Web of Science",
    "isi web of knowledge": "Web of Science",
    "science citation index": "Web of Science",
}

SUGGESTED_CANONICAL_PATTERNS = [

    (re.compile(r"^european nucleotide archive\b|^ena\b", flags=re.IGNORECASE), "European Nucleotide Archive", "ENA spelling/product variant"),
    (re.compile(r"^ribosomal database project\b", flags=re.IGNORECASE), "RDP", "RDP spelling variant"),
    (re.compile(r"^swiss[- ]?prot\b", flags=re.IGNORECASE), "Swiss-Prot", "Swiss-Prot spelling variant"),
    (re.compile(r"^(?:protein data bank|pdb)\b", flags=re.IGNORECASE), "Protein Data Bank", "Protein Data Bank spelling variant"),
    (re.compile(r"^gene ontology\b", flags=re.IGNORECASE), "Gene Ontology", "Gene Ontology spelling variant"),
    (re.compile(r"^world register of marine species\b|^worms\b", flags=re.IGNORECASE), "WoRMS", "WoRMS spelling variant"),
    (re.compile(r"^shuttle radar topography mission\b", flags=re.IGNORECASE), "SRTM", "SRTM expanded-name variant"),
    (re.compile(r"^aster gdem\b", flags=re.IGNORECASE), "ASTER GDEM", "ASTER GDEM spelling/product variant"),
    (re.compile(r"^sentinel[- ]?1\b", flags=re.IGNORECASE), "ESA", "Sentinel-1 operating institution"),
    (re.compile(r"^sentinel[- ]?2\b", flags=re.IGNORECASE), "ESA", "Sentinel-2 operating institution"),
    (re.compile(r"^hadisst\b", flags=re.IGNORECASE), "HadISST", "HadISST spelling/product variant"),
    (re.compile(r"^era[- ]?40\b", flags=re.IGNORECASE), "ERA-40", "ERA-40 spelling/product variant"),
    (re.compile(r"^prism\b", flags=re.IGNORECASE), "PRISM", "PRISM spelling/product variant"),
    (re.compile(r"^aeronet\b", flags=re.IGNORECASE), "AERONET", "AERONET spelling/product variant"),
    (re.compile(r"^avhrr\b", flags=re.IGNORECASE), "AVHRR", "AVHRR spelling/product variant"),
    (re.compile(r"^openstreetmap\b|^open street map\b", flags=re.IGNORECASE), "OpenStreetMap", "OpenStreetMap spelling variant"),
    (re.compile(r"^natural earth\b", flags=re.IGNORECASE), "Natural Earth", "Natural Earth spelling/product variant"),
    (re.compile(r"^ncep/doe reanalysis\b|^ncep doe reanalysis\b", flags=re.IGNORECASE), "NCEP/DOE Reanalysis 2", "NCEP/DOE Reanalysis spelling variant"),
    (re.compile(r"^noaa oi sst\b", flags=re.IGNORECASE), "NOAA OI SST V2", "NOAA OI SST spelling/product variant"),
    (re.compile(r"^science\s*direct\b", flags=re.IGNORECASE), "ScienceDirect", "ScienceDirect spelling variant"),
    (re.compile(r"^isi web of (?:science|knowledge)\b", flags=re.IGNORECASE), "Web of Science", "Web of Science provider-era spelling variant"),
    (re.compile(r"^modis\b", flags=re.IGNORECASE), "MODIS", "MODIS product/family variant"),
    (re.compile(r"^worldclim\b", flags=re.IGNORECASE), "WorldClim", "WorldClim version or product variant"),
    (re.compile(r"^genbank\b", flags=re.IGNORECASE), "GenBank", "GenBank spelling/product variant"),
    (re.compile(r"^ncbi\s+(?:sequence read archive|sra)\b", flags=re.IGNORECASE), "NCBI SRA", "SRA spelling variant"),
    (re.compile(r"^(?:sequence|short) read archive\b", flags=re.IGNORECASE), "NCBI SRA", "SRA spelling variant"),
    (re.compile(r"^era[- ]?interim\b", flags=re.IGNORECASE), "ERA-Interim", "ERA-Interim spelling variant"),
    (re.compile(r"^google earth\b", flags=re.IGNORECASE), "Google Earth", "Google Earth product variant"),
    (re.compile(r"^google maps?\b", flags=re.IGNORECASE), "Google Maps", "Google Maps spelling variant"),
    (re.compile(r"^web of science\b", flags=re.IGNORECASE), "Web of Science", "bibliographic database spelling variant"),
    (re.compile(r"^scopus\b", flags=re.IGNORECASE), "Scopus", "bibliographic database spelling variant"),
    (re.compile(r"^pubmed\b", flags=re.IGNORECASE), "PubMed", "bibliographic database spelling variant"),
    (re.compile(r"^landsat\b", flags=re.IGNORECASE), "Landsat", "Landsat product/family variant"),
    (re.compile(r"^trmm\b", flags=re.IGNORECASE), "TRMM", "TRMM product/family variant"),
    (re.compile(r"^gpcp\b", flags=re.IGNORECASE), "GPCP", "GPCP product/family variant"),
    (re.compile(r"^cmip5\b", flags=re.IGNORECASE), "CMIP5", "CMIP5 experiment/archive variant"),
    (re.compile(r"^cmip6\b", flags=re.IGNORECASE), "CMIP6", "CMIP6 experiment/archive variant"),
]

EARLY_KNOWN_PATTERN_PROPOSALS = [
    (re.compile(r"\bGBIF\b|gbif\.org", flags=re.IGNORECASE), "GBIF", "data_resource_or_product", "GBIF family including URL and slash variants"),
    (re.compile(r"\bWorld Database on Protected Areas\b|\bWorld Database of Protected Areas\b|\bWDPA\b|wdpa\.org", flags=re.IGNORECASE), "World Database on Protected Areas", "data_resource_or_product", "World Database on Protected Areas family"),
    (re.compile(r"\bWorld Database on Other Effective Area[- ]Based Conservation Measures\b|\bWD[- ]?OECM\b|\bWDOECM\b", flags=re.IGNORECASE), "World Database on Other Effective Area-Based Conservation Measures", "data_resource_or_product", "World Database on Other Effective Area-Based Conservation Measures family"),
    (re.compile(r"\bWorld Database (?:of|on) Key Biodiversity Areas\b|\b(?:Global )?Key Biodiversity Areas?\b|^\s*KBAs?\s*$", flags=re.IGNORECASE), "World Database of Key Biodiversity Areas", "data_resource_or_product", "World Database of Key Biodiversity Areas family"),
    (re.compile(r"\b(?:Terrestrial|Marine|Freshwater) Ecoregions? of the World\b|\bWWF\b.*\becoregion(?:s|al)?\b|\becoregionali[sz]ation\b|\b(?:TEOW|MEOW|FEOW)\b", flags=re.IGNORECASE), "WWF Ecoregions", "data_resource_or_product", "WWF ecoregions family"),
    (re.compile(r"\bWorld[- ]?Clim\b|worldclim\.org|worldclim\.com", flags=re.IGNORECASE), "WorldClim", "data_resource_or_product", "WorldClim family"),
    (re.compile(r"^\s*Climate\s*WNA\b", flags=re.IGNORECASE), "ClimateWNA", "data_resource_or_product", "ClimateWNA version family"),
    (re.compile(r"^\s*Climate\s*NA\b", flags=re.IGNORECASE), "ClimateNA", "data_resource_or_product", "ClimateNA version family"),
    (re.compile(r"^\s*Climate\s*EU\b", flags=re.IGNORECASE), "ClimateEU", "data_resource_or_product", "ClimateEU version family"),
    (re.compile(r"^\s*Climate\s*BC\b", flags=re.IGNORECASE), "ClimateBC", "data_resource_or_product", "ClimateBC version family"),
    (re.compile(r"^\s*Climate\s*AP\b", flags=re.IGNORECASE), "ClimateAP", "data_resource_or_product", "ClimateAP version family"),
    (re.compile(r"^(?!.*climatedataguide).*(?:\bClimate-Data(?:\.org)?\b|climate-data\.org|\bclimatedata\.eu\b)", flags=re.IGNORECASE), "Climate-Data.org", "data_resource_or_product", "Climate-Data.org family"),
    (re.compile(r"\bClimate Data Guide\b|climatedataguide\.ucar\.edu", flags=re.IGNORECASE), "NCAR Climate Data Guide", "data_resource_or_product", "NCAR Climate Data Guide family"),
    (re.compile(r"\bClimate Explorer\b", flags=re.IGNORECASE), "KNMI Climate Explorer", "data_resource_or_product", "KNMI Climate Explorer family"),
    (re.compile(r"\b(?:Digital\s+)?Climatic Atlas of the Iberian Peninsula\b", flags=re.IGNORECASE), "Digital Climatic Atlas of the Iberian Peninsula", "data_resource_or_product", "Climatic Atlas of the Iberian Peninsula family"),
    (re.compile(r"\bDigital Climatic Atlas of Catalonia\b", flags=re.IGNORECASE), "Digital Climatic Atlas of Catalonia", "data_resource_or_product", "Digital Climatic Atlas of Catalonia family"),
    (re.compile(r"\bClimate Analysis Section\b", flags=re.IGNORECASE), "NCAR Climate Analysis Section", "data_holder", "NCAR Climate Analysis Section holder family"),
    (re.compile(r"^\s*(?:digital\s+)?(?:elevation|terrain|surface)\s+models?\s*(?:\((?:DEM|DEMs|DTM|DSM)\))?\s*$|^\s*(?:DEM|DTM|DSM)\s*\(\s*Digital\s+(?:Elevation|Terrain|Surface)\s+Model\s*\)\s*$", flags=re.IGNORECASE), "", "generic_or_ambiguous", "generic digital elevation/terrain/surface model label"),
    (re.compile(r"\bDigital Chart of the World\b|^\s*DCW\s*$", flags=re.IGNORECASE), "Digital Chart of the World", "data_resource_or_product", "Digital Chart of the World family"),
    (re.compile(r"^\s*(?:Digital\s*Globe|DigitalGlobe)\b(?!\s*/)", flags=re.IGNORECASE), "DigitalGlobe", "data_holder", "DigitalGlobe/Maxar holder family"),
    (re.compile(r"\bArcGIS\b.*\b(?:World Imagery|imagery basemap|basemap imagery|image(?:ry)? service)\b", flags=re.IGNORECASE), "ArcGIS Online World Imagery", "data_resource_or_product", "ArcGIS imagery/basemap resource family"),
    (re.compile(r"earthexplorer\.usgs\.gov", flags=re.IGNORECASE), "USGS EarthExplorer", "data_resource_or_product", "USGS EarthExplorer domain family"),
    (re.compile(r"\bGoogle\s*Earth\s*Engine\b|earthengine\.google\.com", flags=re.IGNORECASE), "Google Earth Engine", "data_resource_or_product", "Google Earth Engine family"),
    (re.compile(r"\bGoogle\s*Earth\b(?!\s*Engine)", flags=re.IGNORECASE), "Google Earth", "data_resource_or_product", "Google Earth spelling/product family"),
    (re.compile(r"\bGoogle\s*Maps?\b|\bGoogleMaps\b|\bGoogle\s+Satellite(?:\s+(?:images?|imagery|map))?\b", flags=re.IGNORECASE), "Google Maps", "data_resource_or_product", "Google Maps/satellite spelling family"),
    (re.compile(r"\bGoogle\s*Scholar\b|\bGoogleScholar\b|\bGoogle Acad\w*\b|\bGoogle Academic\b", flags=re.IGNORECASE), "Google Scholar", "bibliographic_database", "Google Scholar spelling family"),
    (re.compile(r"\bGoogle Trends\b", flags=re.IGNORECASE), "Google Trends", "data_resource_or_product", "Google Trends family"),
    (re.compile(r"\bGoogle Books\b", flags=re.IGNORECASE), "Google Books", "data_resource_or_product", "Google Books family"),
    (re.compile(r"\bGoogle Images?\b", flags=re.IGNORECASE), "Google Images", "data_resource_or_product", "Google Images family"),
    (re.compile(r"\bGoogle Search(?:es| engine)?\b", flags=re.IGNORECASE), "Google Search", "data_resource_or_product", "Google Search family"),
    (re.compile(r"\bECA\s*&\s*D\b|\bECAD\b|\bEuropean Climate Assessment\s*(?:and|&)\s*Dataset\b", flags=re.IGNORECASE), "ECA&D", "data_resource_or_product", "European Climate Assessment and Dataset family"),
    (re.compile(r"\bEuropean Pollen Database\b", flags=re.IGNORECASE), "European Pollen Database", "data_resource_or_product", "European Pollen Database family"),
    (re.compile(r"\bEuropean Soil Database\b", flags=re.IGNORECASE), "European Soil Database", "data_resource_or_product", "European Soil Database family"),
    (re.compile(r"\bEuropean Fluxes Database(?: Cluster)?\b", flags=re.IGNORECASE), "European Fluxes Database", "data_resource_or_product", "European Fluxes Database family"),
    (re.compile(r"\bEuropean Soil Data Cent(?:er|re)\b|^\s*ESDAC\s*$", flags=re.IGNORECASE), "ESDAC", "data_holder", "European Soil Data Centre holder family"),
    (re.compile(r"\bEuropean Nature Information System\b|^\s*EUNIS\b", flags=re.IGNORECASE), "EUNIS", "data_resource_or_product", "European Nature Information System family"),
    (re.compile(r"\bEuropean Pollutant Release and Transfer Register\b|^\s*E[- ]?PRTR\b", flags=re.IGNORECASE), "E-PRTR", "data_resource_or_product", "European Pollutant Release and Transfer Register family"),
    (re.compile(r"\bEuropean Vegetation Archive\b|^\s*EVA\s*$", flags=re.IGNORECASE), "European Vegetation Archive", "data_resource_or_product", "European Vegetation Archive family"),
    (re.compile(r"\bEuropean Forest Fire Information System\b|^\s*EFFIS\b", flags=re.IGNORECASE), "EFFIS", "data_resource_or_product", "European Forest Fire Information System family"),
    (re.compile(r"\bEuropean Urban Atlas\b", flags=re.IGNORECASE), "European Urban Atlas", "data_resource_or_product", "European Urban Atlas family"),
    (re.compile(r"\bEuropean Register of Marine Species\b|^\s*ERMS\b", flags=re.IGNORECASE), "European Register of Marine Species", "data_resource_or_product", "European Register of Marine Species family"),
    (re.compile(r"\bEuropean Diatom Database\b|^\s*EDDI\b", flags=re.IGNORECASE), "European Diatom Database", "data_resource_or_product", "European Diatom Database family"),
    (re.compile(r"\bChina National Knowledge Infrastructure\b|\bChina Knowledge Resource Integrated Database\b|^\s*CNKI\b", flags=re.IGNORECASE), "CNKI", "bibliographic_database", "CNKI bibliographic database family"),
    (re.compile(r"\bChina Meteorological Forcing Dataset\b|^\s*CMFD\b", flags=re.IGNORECASE), "China Meteorological Forcing Dataset", "data_resource_or_product", "China Meteorological Forcing Dataset family"),
    (re.compile(r"\bChina Meteorological Data Service Cent(?:er|re)\b|^\s*CMDC\b", flags=re.IGNORECASE), "China Meteorological Data Service Center", "data_holder", "China Meteorological Data Service Center holder family"),
    (re.compile(r"\bChina National Environmental Monitoring Cent(?:er|re)\b|^\s*CNEMC\b", flags=re.IGNORECASE), "China National Environmental Monitoring Center", "data_holder", "China National Environmental Monitoring Center holder family"),
    (re.compile(r"\bChina(?:\s+(?:Energy|Rural|Environmental))?\s+Statistical?\s+Yearbooks?\b", flags=re.IGNORECASE), "China Statistical Yearbook", "data_resource_or_product", "China statistical yearbook family"),
    (re.compile(r"^\s*Statistics Canada\b", flags=re.IGNORECASE), "Statistics Canada", "data_holder", "Statistics Canada holder family"),
    (re.compile(r"^\s*Statistics Sweden\b", flags=re.IGNORECASE), "Statistics Sweden", "data_holder", "Statistics Sweden holder family"),
    (re.compile(r"^\s*Statistics Norway\b", flags=re.IGNORECASE), "Statistics Norway", "data_holder", "Statistics Norway holder family"),
    (re.compile(r"^\s*Statistics Netherlands\b", flags=re.IGNORECASE), "Statistics Netherlands", "data_holder", "Statistics Netherlands holder family"),
    (re.compile(r"^\s*Statistics Finland\b", flags=re.IGNORECASE), "Statistics Finland", "data_holder", "Statistics Finland holder family"),
    (re.compile(r"^\s*Statistics Poland\b", flags=re.IGNORECASE), "Statistics Poland", "data_holder", "Statistics Poland holder family"),
    (re.compile(r"^\s*Statistics Korea\b", flags=re.IGNORECASE), "Statistics Korea", "data_holder", "Statistics Korea holder family"),
    (re.compile(r"^\s*(?:Statistics|Statistik) Austria\b", flags=re.IGNORECASE), "Statistics Austria", "data_holder", "Statistics Austria holder family"),
    (re.compile(r"^\s*Statistics Belgium\b", flags=re.IGNORECASE), "Statistics Belgium", "data_holder", "Statistics Belgium holder family"),
    (re.compile(r"^\s*Statistics South Africa\b", flags=re.IGNORECASE), "Statistics South Africa", "data_holder", "Statistics South Africa holder family"),
    (re.compile(r"^\s*Statista\b", flags=re.IGNORECASE), "Statista", "data_holder", "Statista platform/provider family"),
    (re.compile(r"^\s*ForestGEO\b", flags=re.IGNORECASE), "ForestGEO", "data_resource_or_product", "ForestGEO family"),
    (re.compile(r"\bForestPlots(?:\.net)?\b|forestplots\.net", flags=re.IGNORECASE), "ForestPlots.net", "data_resource_or_product", "ForestPlots.net family"),
    (re.compile(r"^\s*forest inventor(?:y|ies)(?:\s+(?:data|database))?\s*$", flags=re.IGNORECASE), "", "generic_or_ambiguous", "generic forest inventory label"),
    (re.compile(r"naturalearthdata\.com|\bNatural\s*Earth\b", flags=re.IGNORECASE), "Natural Earth", "data_resource_or_product", "Natural Earth domain/spelling family"),
    (re.compile(r"\bNatura\s*2000\b", flags=re.IGNORECASE), "Natura 2000", "data_resource_or_product", "Natura 2000 spelling family"),
    (re.compile(r"\bNatureServe\s+(?:Explorer|database)\b", flags=re.IGNORECASE), "NatureServe Explorer", "data_resource_or_product", "NatureServe Explorer family"),
    (re.compile(r"^\s*NatureServe\s*$", flags=re.IGNORECASE), "NatureServe", "data_holder", "NatureServe holder family"),
    (re.compile(r"^\s*Natural England\s*(?:\([A-Z]+\))?\s*$", flags=re.IGNORECASE), "Natural England", "data_holder", "Natural England holder family"),
    (re.compile(r"^\s*Natural Resources Wales\s*(?:\([A-Z]+\))?\s*$", flags=re.IGNORECASE), "Natural Resources Wales", "data_holder", "Natural Resources Wales holder family"),
    (re.compile(r"^\s*Natural Resources Institute Finland\s*(?:\([A-Za-z]+\))?\s*$", flags=re.IGNORECASE), "Natural Resources Institute Finland", "data_holder", "Natural Resources Institute Finland holder family"),
    (re.compile(r"^\s*Natural Resources Canada\s*(?:\([A-Z]+\))?\s*$", flags=re.IGNORECASE), "Natural Resources Canada", "data_holder", "Natural Resources Canada holder family"),
    (re.compile(r"\bUNEP\b.*\bRegional Seas? Programme\b|\bRegional Seas? Programme\b", flags=re.IGNORECASE), "UNEP Regional Seas Programme", "data_holder", "UNEP Regional Seas Programme holder family"),
    (re.compile(r"\bUNCCD\b.*\bPerformance Review and Assessment\b|\bPerformance Review and Assessment of (?:the )?Implementation System\b|\bPRAIS\b", flags=re.IGNORECASE), "UNCCD PRAIS", "data_resource_or_product", "UNCCD PRAIS family"),
    (re.compile(r"\bUN\b.*\b(?:Sustainable Development Goals?|SDG) Indicators? Database\b|\b(?:Global )?SDG Indicators? Database\b|\bUNSD\s*Data\b.*\bSDG\b", flags=re.IGNORECASE), "UN SDG Indicators Database", "data_resource_or_product", "UN SDG Indicators Database family"),
    (re.compile(r"\bUN\b.*\bSustainable Development Goals? Database\b|\bSustainable Development Goals? Database\b", flags=re.IGNORECASE), "UN SDG Indicators Database", "data_resource_or_product", "UN Sustainable Development Goal Database family"),
    (re.compile(r"\bGlobal SDG Indicator Framework\b", flags=re.IGNORECASE), "Global SDG Indicator Framework", "data_resource_or_product", "Global SDG Indicator Framework family"),
    (re.compile(r"\bSystem of Environmental[- ]Economic Accounting\b|\bSEEA\s*(?:EA|EEA|Water)?\b", flags=re.IGNORECASE), "System of Environmental-Economic Accounting", "data_resource_or_product", "SEEA family"),
    (re.compile(r"\bSoil[-\s]*Grids?\b|soilgrids\.org|www\.soilgrids\.org", flags=re.IGNORECASE), "SoilGrids", "data_resource_or_product", "SoilGrids family"),
    (re.compile(r"\bRiver\s*ATLAS\b|\bRiverATLAS\b", flags=re.IGNORECASE), "RiverATLAS", "data_resource_or_product", "RiverATLAS family"),
    (re.compile(r"^\s*Restor\s*$", flags=re.IGNORECASE), "Restor", "data_resource_or_product", "Restor platform family"),
    (re.compile(r"\bRed List of Ecosystems?\b|\bIUCN\b.*\bRed List of Ecosystems?\b", flags=re.IGNORECASE), "IUCN Red List of Ecosystems", "data_resource_or_product", "IUCN Red List of Ecosystems family"),
    (re.compile(r"\bProtected Area Representativeness\b.*\bConnectedness\b", flags=re.IGNORECASE), "Protected Area Representativeness and Connectedness", "data_resource_or_product", "Protected Area Representativeness and Connectedness family"),
    (re.compile(r"\bProject Information Sharing Framework\b", flags=re.IGNORECASE), "Project Information Sharing Framework", "data_resource_or_product", "Project Information Sharing Framework family"),
    (re.compile(r"\bPREDICTS\b", flags=re.IGNORECASE), "PREDICTS", "data_resource_or_product", "PREDICTS database family"),
    (re.compile(r"\bPesticides? Properties Data\s*Base\b|\bPesticides? Properties Database\b|\b(?:FOOTPRINT|IUPAC)\b.*\bPPDB\b|\bPPDB\b.*\b(?:Hertfordshire|pesticides?|aeru)\b|sitem\.herts\.ac\.uk/aeru/ppdb|^\s*PPDB\b", flags=re.IGNORECASE), "Pesticide Properties DataBase", "data_resource_or_product", "Pesticide Properties DataBase family"),
    (re.compile(r"\bOcean Biodiversity Information System\b|\bOcean Biogeographic Information System\b|\bOBIS[- ]?SEAMAP\b|\b(?:Indian Ocean )?Node of OBIS\b|^\s*OBIS\b", flags=re.IGNORECASE), "OBIS", "data_resource_or_product", "Ocean Biodiversity Information System family"),
    (re.compile(r"\bMODIS\b.*\b(?:Vegetation Continuous Fields?|VCF)\b|\bMOD44B\b.*\bVCF\b|\bVegetation Continuous Fields?\b|^\s*VCF\b", flags=re.IGNORECASE), "MODIS Vegetation Continuous Fields", "data_resource_or_product", "MODIS Vegetation Continuous Fields family"),
    (re.compile(r"\bLiving Planet Database\b|\bLiving Planet Database\s*\(LPD\)", flags=re.IGNORECASE), "Living Planet Database", "data_resource_or_product", "Living Planet Database family"),
    (re.compile(r"\bLand[- ]Use Harmoni[sz]ation\b|\bLUH2\b|\bLUH(?:[-\s](?:data(?:\s*set)?|database|project|v\d|CMIP|HYDE|SYNMAP)|\b)|luh\.umd\.edu", flags=re.IGNORECASE), "Land-Use Harmonization", "data_resource_or_product", "Land-Use Harmonization family"),
    (re.compile(r"\bInternational Coral Reef Initiative\b|^\s*ICRI\s*$", flags=re.IGNORECASE), "International Coral Reef Initiative", "data_holder", "International Coral Reef Initiative holder family"),
    (re.compile(r"\bInternational Soil Moisture Network\b|^\s*ISMN\b", flags=re.IGNORECASE), "International Soil Moisture Network", "data_resource_or_product", "International Soil Moisture Network family"),
    (re.compile(r"\bInternational Comprehensive Ocean[- ]Atmosphere Data Set\b|^\s*ICOADS\b", flags=re.IGNORECASE), "ICOADS", "data_resource_or_product", "International Comprehensive Ocean-Atmosphere Data Set family"),
    (re.compile(r"\bInternational Satellite Cloud Climatology Project\b|^\s*ISCCP\b", flags=re.IGNORECASE), "ISCCP", "data_resource_or_product", "International Satellite Cloud Climatology Project family"),
    (re.compile(r"\bInternational Tree[- ]Ring Data Bank\b|^\s*ITRDB\b", flags=re.IGNORECASE), "International Tree-Ring Data Bank", "data_resource_or_product", "International Tree-Ring Data Bank family"),
    (re.compile(r"\bInternational Best Track Archive for Climate Stewardship\b|^\s*IBTrACS\b", flags=re.IGNORECASE), "IBTrACS", "data_resource_or_product", "International Best Track Archive for Climate Stewardship family"),
    (re.compile(r"\bInternational Waterbird Census\b|^\s*IWC\s*$", flags=re.IGNORECASE), "International Waterbird Census", "data_resource_or_product", "International Waterbird Census family"),
    (re.compile(r"\bWorld Economic Outlook\b|^\s*WEO\b", flags=re.IGNORECASE), "IMF World Economic Outlook", "data_resource_or_product", "IMF World Economic Outlook family"),
    (re.compile(r"\bInternational Monetary Fund\b|^\s*IMF\s*$", flags=re.IGNORECASE), "International Monetary Fund", "data_holder", "International Monetary Fund holder family"),
    (re.compile(r"\bCopernicus Climate Change Service\b|^\s*C3S\b", flags=re.IGNORECASE), "Copernicus Climate Change Service", "data_resource_or_product", "Copernicus Climate Change Service family"),
    (re.compile(r"\bCopernicus Global Land Service\b|^\s*CGLS\b", flags=re.IGNORECASE), "Copernicus Global Land Service", "data_resource_or_product", "Copernicus Global Land Service family"),
    (re.compile(r"\bCopernicus Land Monitoring Services?\b|^\s*CLMS\b", flags=re.IGNORECASE), "Copernicus Land Monitoring Service", "data_resource_or_product", "Copernicus Land Monitoring Service family"),
    (re.compile(r"\bCopernicus Open Access Hub\b|\bCopernicus Online Data Access\b|^\s*CODA\b", flags=re.IGNORECASE), "Copernicus Open Access Hub", "data_resource_or_product", "Copernicus Open Access Hub/CODA family"),
    (re.compile(r"\bCopernicus Data and Information Access Services\b|^\s*DIAS\b", flags=re.IGNORECASE), "Copernicus DIAS", "data_resource_or_product", "Copernicus DIAS family"),
    (re.compile(r"\bOcean\s*Colou?r\s+Web\b|\bNASA\b.*\bOcean\s*Colou?r\s+Web\b|oceancolor\.gsfc\.nasa\.gov", flags=re.IGNORECASE), "NASA Ocean Color Web", "data_resource_or_product", "NASA Ocean Color Web family"),
    (re.compile(r"\bOcean\s*Colou?r\s+(?:website|web site|web page|web portal|web server)\b", flags=re.IGNORECASE), "NASA Ocean Color Web", "data_resource_or_product", "NASA Ocean Color Web family"),
    (re.compile(r"\bOcean\s*Colou?r\s+Climate Change Initiative\b|^\s*OC[-_ ]?CCI\b", flags=re.IGNORECASE), "Ocean Colour CCI", "data_resource_or_product", "Ocean Colour CCI family"),
    (re.compile(r"\bOcean Data Viewer\b", flags=re.IGNORECASE), "Ocean Data View", "software_or_tool", "Ocean Data View/Viewer family"),
    (re.compile(r"\bOceanSITES\b", flags=re.IGNORECASE), "OceanSITES", "data_resource_or_product", "OceanSITES family"),
    (re.compile(r"\bOcean Productivity\s*$|\bOcean Productivity\s+(?:website|site|data|database|dataset|project)\b|science\.oregonstate\.edu/ocean\.productivity", flags=re.IGNORECASE), "Ocean Productivity", "data_resource_or_product", "Ocean Productivity family"),
    (re.compile(r"\bOcean Health Index\b", flags=re.IGNORECASE), "Ocean Health Index", "data_resource_or_product", "Ocean Health Index family"),
    (re.compile(r"\bOcean Tracking Network\b|^\s*OTN\b", flags=re.IGNORECASE), "Ocean Tracking Network", "data_resource_or_product", "Ocean Tracking Network family"),
    (re.compile(r"\bPlantCyc\b", flags=re.IGNORECASE), "PlantCyc", "data_resource_or_product", "PlantCyc family"),
    (re.compile(r"\bPlantGDB\b", flags=re.IGNORECASE), "PlantGDB", "data_resource_or_product", "PlantGDB family"),
    (re.compile(r"^\s*Plant List\b|\bThe Plant List\b", flags=re.IGNORECASE), "The Plant List", "data_resource_or_product", "The Plant List family"),
    (re.compile(r"(?:\bUSDA\b|\bUSDA[-\s]?NRCS\b|\bNRCS\b).*\bPLANTS Database\b|^\s*PLANTS Database\b|plants\.usda\.gov", flags=re.IGNORECASE), "USDA PLANTS Database", "data_resource_or_product", "USDA PLANTS Database family"),
    (re.compile(r"\bPlant Transcription Factor Database\b|\bPlantTFDB\b", flags=re.IGNORECASE), "PlantTFDB", "data_resource_or_product", "PlantTFDB family"),
    (re.compile(r"\bPlant Genome Duplication Database\b|^\s*PGDD\b", flags=re.IGNORECASE), "Plant Genome Duplication Database", "data_resource_or_product", "Plant Genome Duplication Database family"),
    (re.compile(r"\bPlantCARE\b", flags=re.IGNORECASE), "PlantCARE", "data_resource_or_product", "PlantCARE family"),
    (re.compile(r"\bPlants of the World Online\b|^\s*POWO\b", flags=re.IGNORECASE), "Plants of the World Online", "data_resource_or_product", "Plants of the World Online family"),
    (re.compile(r"\bPlant DNA C[- ]values? database\b", flags=re.IGNORECASE), "Plant DNA C-values Database", "data_resource_or_product", "Plant DNA C-values Database family"),
    (re.compile(r"\bPlant Metabolic Network\b|^\s*PMN\b", flags=re.IGNORECASE), "Plant Metabolic Network", "data_resource_or_product", "Plant Metabolic Network family"),
    (re.compile(r"\bMarine\s*04\b|\bMARINE04\b", flags=re.IGNORECASE), "Marine04", "data_resource_or_product", "Marine04 calibration curve family"),
    (re.compile(r"\bMarine\s*09\b|\bMARINE09\b", flags=re.IGNORECASE), "Marine09", "data_resource_or_product", "Marine09 calibration curve family"),
    (re.compile(r"\bMarine\s*13\b|\bMARINE13\b", flags=re.IGNORECASE), "Marine13", "data_resource_or_product", "Marine13 calibration curve family"),
    (re.compile(r"\bMarine\s*20\b|\bMARINE20\b", flags=re.IGNORECASE), "Marine20", "data_resource_or_product", "Marine20 calibration curve family"),
    (re.compile(r"\bMarine Reservoir (?:Correction )?Database\b", flags=re.IGNORECASE), "Marine Reservoir Correction Database", "data_resource_or_product", "Marine Reservoir Correction Database family"),
    (re.compile(r"\bMarine\s*Traffic\b|\bMarineTraffic\b", flags=re.IGNORECASE), "MarineTraffic", "data_resource_or_product", "MarineTraffic family"),
    (re.compile(r"\bMarine Regions\b|marineregions\.org", flags=re.IGNORECASE), "Marine Regions", "data_resource_or_product", "Marine Regions family"),
    (re.compile(r"\bMarine Geoscience Data System\b", flags=re.IGNORECASE), "Marine Geoscience Data System", "data_resource_or_product", "Marine Geoscience Data System family"),
    (re.compile(r"\bMarine Recreational Information Program\b|^\s*MRIP\b", flags=re.IGNORECASE), "Marine Recreational Information Program", "data_resource_or_product", "Marine Recreational Information Program family"),
    (re.compile(r"marine\.copernicus\.eu", flags=re.IGNORECASE), "Copernicus Marine Service", "data_resource_or_product", "Copernicus Marine Service domain family"),
    (re.compile(r"\bFlora do Brasil\b|\bLista de Esp.cies da Flora do Brasil\b", flags=re.IGNORECASE), "Flora do Brasil", "data_resource_or_product", "Flora do Brasil family"),
    (re.compile(r"\bFlora of North America\b", flags=re.IGNORECASE), "Flora of North America", "data_resource_or_product", "Flora of North America family"),
    (re.compile(r"\bFlora of Pakistan\b", flags=re.IGNORECASE), "Flora of Pakistan", "data_resource_or_product", "Flora of Pakistan family"),
    (re.compile(r"\bFlora of China\b|\bFlora Reipublicae Popularis Sinicae\b", flags=re.IGNORECASE), "Flora of China", "data_resource_or_product", "Flora of China family"),
    (re.compile(r"\bFlora Europ(?:a|ae)a\b|\bFlora Europea\b", flags=re.IGNORECASE), "Flora Europaea", "data_resource_or_product", "Flora Europaea family"),
    (re.compile(r"\bFlora Iberica\b", flags=re.IGNORECASE), "Flora Iberica", "data_resource_or_product", "Flora Iberica family"),
    (re.compile(r"\bFloraWeb\b", flags=re.IGNORECASE), "FloraWeb", "data_resource_or_product", "FloraWeb family"),
    (re.compile(r"\bFloraBase\b", flags=re.IGNORECASE), "FloraBase", "data_resource_or_product", "FloraBase family"),
    (re.compile(r"\bFlora Iranica\b|\bFlora of Iran\b", flags=re.IGNORECASE), "Flora Iranica", "data_resource_or_product", "Flora Iranica family"),
    (re.compile(r"\bFlora Malesiana\b", flags=re.IGNORECASE), "Flora Malesiana", "data_resource_or_product", "Flora Malesiana family"),
    (re.compile(r"\bUniRef(?:50|90|100)?\b", flags=re.IGNORECASE), "UniRef", "data_resource_or_product", "UniRef family"),
    (re.compile(r"\bUniProt[- ]GOA\b", flags=re.IGNORECASE), "UniProt-GOA", "data_resource_or_product", "UniProt-GOA family"),
    (re.compile(r"\bUniProt[_-]sprot\b", flags=re.IGNORECASE), "Swiss-Prot", "data_resource_or_product", "Swiss-Prot family"),
    (re.compile(r"\bUniProt(?:KB)?\s*/?\s*Swiss[- ]?Prot\b|\bSwiss[- ]?Prot\b", flags=re.IGNORECASE), "Swiss-Prot", "data_resource_or_product", "Swiss-Prot family"),
    (re.compile(r"\bUniProt(?:KB)?\s*/?\s*TrEMBL\b|\bTrEMBL\b", flags=re.IGNORECASE), "TrEMBL", "data_resource_or_product", "TrEMBL family"),
    (re.compile(r"\bUniProt(?:KB|\s+KB|\s+Knowledgebase|\s+KnowledgeBase|\s+protein database)?\b", flags=re.IGNORECASE), "UniProt", "data_resource_or_product", "UniProt family"),
    (re.compile(r"\bGlobal Register (?:of|for) Introduced and Invasive Species\b|^\s*GRIIS\b|griis\.org", flags=re.IGNORECASE), "Global Register of Introduced and Invasive Species", "data_resource_or_product", "Global Register of Introduced and Invasive Species family"),
    (re.compile(r"\bGlobal Invasive Species Database\b|^\s*GISD\b", flags=re.IGNORECASE), "Global Invasive Species Database", "data_resource_or_product", "Global Invasive Species Database family"),
    (re.compile(r"\bGlobal Names Index\b|^\s*GNI\b", flags=re.IGNORECASE), "Global Names Index", "data_resource_or_product", "Global Names Index family"),
    (re.compile(r"\bGlobal Carbon Project\b", flags=re.IGNORECASE), "Global Carbon Project", "data_holder", "Global Carbon Project holder/project family"),
    (re.compile(r"\bGlobal Fishing Watch\b", flags=re.IGNORECASE), "Global Fishing Watch", "data_holder", "Global Fishing Watch holder/platform family"),
    (re.compile(r"\bGlobal Plants\b", flags=re.IGNORECASE), "Global Plants", "data_resource_or_product", "Global Plants/JSTOR family"),
    (re.compile(r"\bGlobal Precipitation Climatology Project\b", flags=re.IGNORECASE), "GPCP", "data_resource_or_product", "Global Precipitation Climatology Project family"),
    (re.compile(r"\bGlobal Pesticide Grids?\b", flags=re.IGNORECASE), "Global Pesticide Grids", "data_resource_or_product", "Global Pesticide Grids family"),
    (re.compile(r"\bGlobal Human Settlement Layer\b|^\s*GHSL\b", flags=re.IGNORECASE), "Global Human Settlement Layer", "data_resource_or_product", "Global Human Settlement Layer family"),
    (re.compile(r"\bGlobal Land Cover Characteri[sz]ation\b|^\s*GLCC\b", flags=re.IGNORECASE), "Global Land Cover Characterization", "data_resource_or_product", "Global Land Cover Characterization family"),
    (re.compile(r"\bGlobal Agro[- ]Ecological Zones\b|^\s*GAEZ\b", flags=re.IGNORECASE), "Global Agro-Ecological Zones", "data_resource_or_product", "Global Agro-Ecological Zones family"),
    (re.compile(r"\bFiner Resolution Observation and Monitoring of Global Land Cover\b|^\s*FROM[- ]?GLC\b", flags=re.IGNORECASE), "FROM-GLC", "data_resource_or_product", "FROM-GLC family"),
    (re.compile(r"\bGlobal Land Cover SHARE\b|\bGLC[- ]?SHARE\b", flags=re.IGNORECASE), "GLC-SHARE", "data_resource_or_product", "GLC-SHARE family"),
    (re.compile(r"\bWorldPop\b", flags=re.IGNORECASE), "WorldPop", "data_resource_or_product", "WorldPop family"),
    (re.compile(r"\bWorld Economic Outlook\b|^\s*WEO\b", flags=re.IGNORECASE), "IMF World Economic Outlook", "data_resource_or_product", "IMF World Economic Outlook family"),
    (re.compile(r"\bWorld Population Prospects\b|^\s*WPP\b", flags=re.IGNORECASE), "World Population Prospects", "data_resource_or_product", "World Population Prospects family"),
    (re.compile(r"\bHarmoni[sz]ed World Soil Database\b|^\s*HWSD\b", flags=re.IGNORECASE), "Harmonized World Soil Database", "data_resource_or_product", "Harmonized World Soil Database family"),
    (re.compile(r"\bWorld Ocean Atlas\b|^\s*WOA\d*\b", flags=re.IGNORECASE), "World Ocean Atlas", "data_resource_or_product", "World Ocean Atlas family"),
    (re.compile(r"\bNational Biodiversity Network\b|^\s*NBN\b", flags=re.IGNORECASE), "National Biodiversity Network", "data_resource_or_product", "National Biodiversity Network family"),
    (re.compile(r"\bNational Ecological Observatory Network\b|^\s*NEON\b", flags=re.IGNORECASE), "National Ecological Observatory Network", "data_resource_or_product", "National Ecological Observatory Network family"),
    (re.compile(r"\bNational Atmospheric Deposition Program\b|^\s*NADP\b", flags=re.IGNORECASE), "National Atmospheric Deposition Program", "data_resource_or_product", "National Atmospheric Deposition Program family"),
    (re.compile(r"\bNHD\s*Plus\b|\bNHDPlus\b|\bNational Hydrography Dataset\b.*\bPlus\b", flags=re.IGNORECASE), "NHDPlus", "data_resource_or_product", "NHDPlus family"),
    (re.compile(r"\bNational Hydrography Dataset\b|^\s*NHD\b", flags=re.IGNORECASE), "National Hydrography Dataset", "data_resource_or_product", "National Hydrography Dataset family"),
    (re.compile(r"\bNational Water Information System\b|^\s*NWIS\b", flags=re.IGNORECASE), "USGS NWIS", "data_resource_or_product", "USGS National Water Information System family"),
    (re.compile(r"\bGlobal Forest Resources? Assessment\b|\bFAO(?:'s)?\b.*\bForest Resources? Assessment\b|\bForest Resources? Assessment\b.*\bFAO\b|\bForest Resources? Assessment\s*\(FRA\)|\bFRA\s*20\d{2}\b", flags=re.IGNORECASE), "FAO FRA", "data_resource_or_product", "FAO Forest Resources Assessment family"),
    (re.compile(r"\bGlobal Ecosystems? Atlas\b", flags=re.IGNORECASE), "Global Ecosystems Atlas", "data_resource_or_product", "Global Ecosystems Atlas family"),
    (re.compile(r"\bGlobal Database on Protected Area Management Effectiveness\b|\bGD[- ]?PAME\b|\bGDPAME\b", flags=re.IGNORECASE), "Global Database on Protected Area Management Effectiveness", "data_resource_or_product", "Global Database on Protected Area Management Effectiveness family"),
    (re.compile(r"\bGlobal Biodiversity Change Indicators?\b|^\s*GBCI\b", flags=re.IGNORECASE), "Global Biodiversity Change Indicators", "data_resource_or_product", "Global Biodiversity Change Indicators family"),
    (re.compile(r"\bFramework for Ecosystem Restoration Monitoring\b|^\s*FERM\b", flags=re.IGNORECASE), "Framework for Ecosystem Restoration Monitoring", "data_resource_or_product", "Framework for Ecosystem Restoration Monitoring family"),
    (re.compile(r"\bForest Landscape Integrity Index\b|^\s*FLII\b", flags=re.IGNORECASE), "Forest Landscape Integrity Index", "data_resource_or_product", "Forest Landscape Integrity Index family"),
    (re.compile(r"\bFAO Statistical Database\b|faostat\.fao\.org", flags=re.IGNORECASE), "FAOSTAT", "data_resource_or_product", "FAOSTAT family"),
    (re.compile(r"\bESA\b.*\bLand[- ]Cover\b|\bESA[- ]?CCI\b.*\bLand[- ]Cover\b|\bClimate Change Initiative\b.*\bLand[- ]Cover\b|\bCCI[-_ ]?LC\b|\bLC[-_ ]?CCI\b", flags=re.IGNORECASE), "ESA Land Cover", "data_resource_or_product", "ESA Land Cover family"),
    (re.compile(r"\bEcosystem Intactness Index\b", flags=re.IGNORECASE), "Ecosystem Intactness Index", "data_resource_or_product", "Ecosystem Intactness Index family"),
    (re.compile(r"\bEarth Environment\s*\(EarthEnv\)", flags=re.IGNORECASE), "EarthEnv", "data_resource_or_product", "EarthEnv family"),
    (re.compile(r"\bCSIRO\b.*\bBiodiversity Indicators Explorer\b|\bBiodiversity Indicators Explorer\b", flags=re.IGNORECASE), "CSIRO Biodiversity Indicators Explorer", "data_resource_or_product", "CSIRO Biodiversity Indicators Explorer family"),
    (re.compile(r"\bCSIRO\b.*\bData Access Portal\b", flags=re.IGNORECASE), "CSIRO Data Access Portal", "data_resource_or_product", "CSIRO Data Access Portal family"),
    (re.compile(r"\bCROP\s*GRIDS?\b|\bCROPGRIDS\b", flags=re.IGNORECASE), "CROPGRIDS", "data_resource_or_product", "CROPGRIDS family"),
    (re.compile(r"\bCBD\b.*\bGlobal\b.*\bCHM\b|\bConvention on Biological Diversity\b.*\bClearing[- ]House Mechanism\b|\bClearing[- ]House Mechanism\b.*\bConvention on Biological Diversity\b|\bGlobal\b.*\bClearing[- ]House Mechanism\b", flags=re.IGNORECASE), "CBD Global CHM", "data_resource_or_product", "CBD Global CHM family"),
    (re.compile(r"\bBiodiversity Intactness Index\b", flags=re.IGNORECASE), "Biodiversity Intactness Index", "data_resource_or_product", "Biodiversity Intactness Index family"),
    (re.compile(r"\bBiodiversity Indicators Partnership Dashboard\b|\bBIP Dashboard\b", flags=re.IGNORECASE), "Biodiversity Indicators Partnership Dashboard", "data_resource_or_product", "Biodiversity Indicators Partnership Dashboard family"),
    (re.compile(r"\bBiodiversity Habitat Index\b", flags=re.IGNORECASE), "Biodiversity Habitat Index", "data_resource_or_product", "Biodiversity Habitat Index family"),
    (re.compile(r"\bBioclimatic Ecosystem Resilience Index\b", flags=re.IGNORECASE), "Bioclimatic Ecosystem Resilience Index", "data_resource_or_product", "Bioclimatic Ecosystem Resilience Index family"),
    (re.compile(r"\bAlien Species First Record Database\b|\bGlobal Alien Species First Record Database\b", flags=re.IGNORECASE), "Alien Species First Record Database", "data_resource_or_product", "Alien Species First Record Database family"),
    (re.compile(r"^SILVA\b", flags=re.IGNORECASE), "SILVA", "data_resource_or_product", "SILVA database/version family"),
    (re.compile(r"\b(?:ARB|Mothur)[-\s]?SILVA\b|\bSILVA\b.*\b(?:database|databases|taxonomy|taxonomic|SSU|LSU|SSURef|LSURef|rRNA|ribosomal|reference|alignment|seed|nr|release|classifier)\b|\b(?:database|databases|taxonomy|taxonomic|SSU|LSU|SSURef|LSURef|rRNA|ribosomal|reference|alignment|seed|nr|release|classifier)\b.*\bSILVA\b", flags=re.IGNORECASE), "SILVA", "data_resource_or_product", "SILVA database/context family"),
    (re.compile(r"\bCORINE\b.*\b(?:land|cover|CLC)\b|^CORINE\b", flags=re.IGNORECASE), "CORINE Land Cover", "data_resource_or_product", "CORINE Land Cover family"),
    (re.compile(r"^GADM\b|\bGlobal Administrative Areas\b", flags=re.IGNORECASE), "GADM", "data_resource_or_product", "GADM administrative boundary family"),
    (re.compile(r"\b(?:Copernicus Marine|CMEMS|Marine Environment Monitoring Service)\b", flags=re.IGNORECASE), "Copernicus Marine Service", "data_resource_or_product", "Copernicus Marine Service family"),
    (re.compile(r"\b(?:Copernicus Atmosphere|CAMS)\b", flags=re.IGNORECASE), "Copernicus Atmosphere Monitoring Service", "data_resource_or_product", "Copernicus Atmosphere Monitoring Service family"),
    (re.compile(r"\b(?:Copernicus Climate Data Store|Climate Data Store|CDS)\b", flags=re.IGNORECASE), "Copernicus Climate Data Store", "data_resource_or_product", "Copernicus Climate Data Store family"),
    (re.compile(r"\bCopernicus EMS\b|\bEmergency Management Service\b", flags=re.IGNORECASE), "Copernicus EMS", "data_resource_or_product", "Copernicus Emergency Management Service family"),
    (re.compile(r"\bESA\b.*\bGlobCover\b|^GlobCover\b", flags=re.IGNORECASE), "ESA GlobCover", "data_resource_or_product", "ESA GlobCover family"),
    (re.compile(r"\bESA\b.*\bSMOS\b|^SMOS\b", flags=re.IGNORECASE), "ESA SMOS", "data_resource_or_product", "ESA SMOS family"),
    (re.compile(r"\bGES\s*DISC\b|\bGoddard Earth Sciences Data and Information Services Center\b", flags=re.IGNORECASE), "NASA GES DISC", "data_resource_or_product", "NASA GES DISC family"),
    (re.compile(r"\b(?:NASA\s+)?(?:Earthdata|EOSDIS|Earth Observing System Data and Information System)\b", flags=re.IGNORECASE), "NASA Earthdata/EOSDIS", "data_resource_or_product", "NASA Earthdata/EOSDIS family"),
    (re.compile(r"^JPL\b|\bJet Propulsion Laboratory\b", flags=re.IGNORECASE), "NASA JPL", "data_holder", "NASA Jet Propulsion Laboratory holder family"),
    (re.compile(r"\b(?:NOAA\s+)?GDAS\b|\bGlobal Data Assimilation System\b", flags=re.IGNORECASE), "NOAA GDAS", "data_resource_or_product", "NOAA GDAS family"),
    (re.compile(r"\b(?:NOAA\s+|NCEP\s+)?GFS\b|\bGlobal Forecast System\b", flags=re.IGNORECASE), "NOAA GFS", "data_resource_or_product", "NOAA GFS family"),
    (re.compile(r"\b(?:NOAA\s+|NCEP\s+)?FNL\b|\bFinal Operational Global Analysis\b", flags=re.IGNORECASE), "NOAA FNL", "data_resource_or_product", "NOAA FNL family"),
    (re.compile(r"\b(?:NOAA\s+|NCEP\s+)?GODAS\b|\bGlobal Ocean Data Assimilation System\b", flags=re.IGNORECASE), "NOAA GODAS", "data_resource_or_product", "NOAA GODAS family"),
    (re.compile(r"\bNARR\b|\bNorth American Regional Reanalysis\b", flags=re.IGNORECASE), "NOAA NARR", "data_resource_or_product", "NOAA NARR family"),
    (re.compile(r"\b(?:NOAA\s+)?NGDC\b|\bNational Geophysical Data Center\b", flags=re.IGNORECASE), "NOAA NGDC", "data_holder", "NOAA NGDC holder family"),
    (re.compile(r"\b(?:NOAA\s+)?NODC\b|\bNational Oceanographic Data Center\b|\bNational Oceanographic Data Centre\b", flags=re.IGNORECASE), "NOAA NODC", "data_holder", "NOAA NODC holder family"),
    (re.compile(r"\b(?:NOAA\s+)?NDBC\b|\bNational Data Buoy Center\b", flags=re.IGNORECASE), "NOAA NDBC", "data_holder", "NOAA NDBC holder family"),
    (re.compile(r"\bNOAA\b.*\bGlobal Monitoring Laboratory\b|\bNOAA\b.*\bGML\b", flags=re.IGNORECASE), "NOAA Global Monitoring Laboratory", "data_holder", "NOAA Global Monitoring Laboratory holder family"),
    (re.compile(r"\bETOPO(?:1|2|5)?\b", flags=re.IGNORECASE), "NOAA ETOPO", "data_resource_or_product", "NOAA ETOPO topography family"),
    (re.compile(r"\bScienceBase\b", flags=re.IGNORECASE), "USGS ScienceBase", "data_resource_or_product", "USGS ScienceBase family"),
    (re.compile(r"\bUSGS\b.*\bEROS\b|\bEarth Resources Observation and Science Center\b", flags=re.IGNORECASE), "USGS EROS", "data_holder", "USGS EROS holder family"),
    (re.compile(r"\bGMTED2010\b|\bGlobal Multi-resolution Terrain Elevation Data\b", flags=re.IGNORECASE), "USGS GMTED2010", "data_resource_or_product", "USGS GMTED2010 family"),
    (re.compile(r"\bBird Banding Laboratory\b", flags=re.IGNORECASE), "USGS Bird Banding Laboratory", "data_holder", "USGS Bird Banding Laboratory holder family"),
    (re.compile(r"\b(?:US\s*EPA|USEPA|U\.S\.\s*EPA)\b$|^EPA\b$", flags=re.IGNORECASE), "US EPA", "data_holder", "US EPA holder family"),
    (re.compile(r"\b(?:EPA\s+)?AQS\b|\bAir Quality System\b", flags=re.IGNORECASE), "EPA AQS", "data_resource_or_product", "EPA Air Quality System family"),
    (re.compile(r"\bECOTOX\b", flags=re.IGNORECASE), "EPA ECOTOX", "data_resource_or_product", "EPA ECOTOX family"),
    (re.compile(r"\b(?:EPA|US\s*EPA|USEPA|U\.S\.\s*EPA)\b.*\b(?:National Emissions Inventory|NEI)\b|^\s*National Emissions Inventory\s*(?:\(|$|database\b|dataset\b)|^\s*NEI\s*(?:\(|$|database\b|dataset\b)", flags=re.IGNORECASE), "EPA National Emissions Inventory", "data_resource_or_product", "EPA National Emissions Inventory family"),
    (re.compile(r"\b(?:EPA\s+)?IRIS\b|\bIntegrated Risk Information System\b", flags=re.IGNORECASE), "EPA IRIS", "data_resource_or_product", "EPA IRIS family"),
    (re.compile(r"\bEPA\b.*\b(?:Level\s*(?:III|IV|3|4)\s+)?ecoregions?\b", flags=re.IGNORECASE), "EPA Ecoregions", "data_resource_or_product", "EPA ecoregions family"),
    (re.compile(r"\bEPA\b.*\bmethod\s*\d+", flags=re.IGNORECASE), "", "false_positive", "EPA laboratory method, not reusable data source"),
    (re.compile(r"\bSSURGO\b|\bSoil Survey Geographic\b", flags=re.IGNORECASE), "USDA SSURGO", "data_resource_or_product", "USDA SSURGO family"),
    (re.compile(r"\b(?:Cropland Data Layer|CDL)\b", flags=re.IGNORECASE), "USDA Cropland Data Layer", "data_resource_or_product", "USDA Cropland Data Layer family"),
    (re.compile(r"\bNAIP\b|\bNational Agriculture Imagery Program\b", flags=re.IGNORECASE), "USDA NAIP", "data_resource_or_product", "USDA NAIP family"),
    (re.compile(r"\bUSDA\b.*\b(?:National Nutrient Database|FoodData Central|nutrient database)\b", flags=re.IGNORECASE), "USDA National Nutrient Database", "data_resource_or_product", "USDA nutrient database family"),
    (re.compile(r"\bUSDA\b.*\bNRCS\b|\bNatural Resources Conservation Service\b", flags=re.IGNORECASE), "USDA NRCS", "data_holder", "USDA NRCS holder family"),
    (re.compile(r"\bUSDA\b.*\bNASS\b|\bNational Agricultural Statistics Service\b", flags=re.IGNORECASE), "USDA NASS", "data_holder", "USDA NASS holder family"),
    (re.compile(r"\bUSDA\b.*\bERS\b|\bEconomic Research Service\b", flags=re.IGNORECASE), "USDA ERS", "data_holder", "USDA ERS holder family"),
    (re.compile(r"\bAQUASTAT\b", flags=re.IGNORECASE), "FAO AQUASTAT", "data_resource_or_product", "FAO AQUASTAT family"),
    (re.compile(r"\bFishStat\b", flags=re.IGNORECASE), "FAO FishStat", "data_resource_or_product", "FAO FishStat family"),
    (re.compile(r"\bFood Balance Sheets?\b", flags=re.IGNORECASE), "FAO Food Balance Sheets", "data_resource_or_product", "FAO Food Balance Sheets family"),
    (re.compile(r"\bFAO\b.*\bGeoNetwork\b", flags=re.IGNORECASE), "FAO GeoNetwork", "data_resource_or_product", "FAO GeoNetwork family"),
    (re.compile(r"\bFAO\b.*\bSoil Map of the World\b|\bDigital Soil Map of the World\b", flags=re.IGNORECASE), "FAO Soil Map of the World", "data_resource_or_product", "FAO Soil Map of the World family"),
    (re.compile(r"\bNIST\b.*\b(?:SRM|Standard Reference Material)\b", flags=re.IGNORECASE), "NIST Standard Reference Materials", "data_resource_or_product", "NIST Standard Reference Materials family"),
    (re.compile(r"\bNIST\b.*\b(?:Mass Spectral|MS/MS|library|Libraries)\b", flags=re.IGNORECASE), "NIST Mass Spectral Library", "data_resource_or_product", "NIST Mass Spectral Library family"),
    (re.compile(r"\bUK\b.*\bLand Cover Map\b", flags=re.IGNORECASE), "UK Land Cover Map", "data_resource_or_product", "UK Land Cover Map family"),
    (re.compile(r"\bUK\b.*\bCountryside Survey\b", flags=re.IGNORECASE), "UK Countryside Survey", "data_resource_or_product", "UK Countryside Survey family"),
    (re.compile(r"\bOrdnance Survey\b.*\bMaster\s*Map\b|\bMasterMap\b", flags=re.IGNORECASE), "UK Ordnance Survey MasterMap", "data_resource_or_product", "Ordnance Survey MasterMap family"),
    (re.compile(r"\bNational River Flow Archive\b|\bNRFA\b", flags=re.IGNORECASE), "UK National River Flow Archive", "data_resource_or_product", "UK National River Flow Archive family"),
    (re.compile(r"\b(?:Met Office|Hadley Centre|Hadley Center)\b", flags=re.IGNORECASE), "UK Met Office Hadley Centre", "data_holder", "UK Met Office/Hadley Centre holder family"),
    (re.compile(r"\bEnvironment Canada\b|\bEnvironment and Climate Change Canada\b|\bECCC\b", flags=re.IGNORECASE), "Environment Canada", "data_holder", "Environment Canada holder family"),
    (re.compile(r"^Environment Agency\b", flags=re.IGNORECASE), "Environment Agency", "data_holder", "Environment Agency holder family"),
    (re.compile(r"\b(?:European Chemicals Agency|ECHA)\b", flags=re.IGNORECASE), "ECHA", "data_holder", "European Chemicals Agency holder family"),
    (re.compile(r"\b(?:European Commission Joint Research Centre|Joint Research Centre|\bJRC\b)\b", flags=re.IGNORECASE), "European Commission JRC", "data_holder", "European Commission JRC holder family"),
    (re.compile(r"\b(?:EMODnet|European Marine Observation and Data Network)\b", flags=re.IGNORECASE), "EMODnet", "data_resource_or_product", "EMODnet family"),
    (re.compile(r"\bEuropean Fire Database\b", flags=re.IGNORECASE), "European Fire Database", "data_resource_or_product", "European Fire Database family"),
    (re.compile(r"\b(?:European Genome-phenome Archive|European Genome Phenome Archive|\bEGA\b)\b", flags=re.IGNORECASE), "EGA", "data_resource_or_product", "European Genome-phenome Archive family"),
    (re.compile(r"^EMEP\b|\bEuropean Monitoring and Evaluation Programme\b", flags=re.IGNORECASE), "EMEP", "data_resource_or_product", "EMEP family"),
    (re.compile(r"^WorldPop\b|\bWorldPop\b", flags=re.IGNORECASE), "WorldPop", "data_resource_or_product", "WorldPop family"),
    (re.compile(r"\bGridded Population of the World\b|^GPW\b", flags=re.IGNORECASE), "GPW", "data_resource_or_product", "Gridded Population of the World family"),
    (re.compile(r"^HYDE\b|\bHistory Database of the Global Environment\b", flags=re.IGNORECASE), "HYDE", "data_resource_or_product", "HYDE family"),
    (re.compile(r"\b(?:Global Historical Climatology Network|Global Historical Climate Network|\bGHCN\b)", flags=re.IGNORECASE), "GHCN", "data_resource_or_product", "GHCN family"),
    (re.compile(r"\bGLDAS\b|\bGlobal Land Data Assimilation System\b", flags=re.IGNORECASE), "GLDAS", "data_resource_or_product", "GLDAS family"),
    (re.compile(r"\bNLDAS\b|\bNorth American Land Data Assimilation System\b", flags=re.IGNORECASE), "NLDAS", "data_resource_or_product", "NLDAS family"),
    (re.compile(r"\bGlobal Surface Water\b|^GSW\b", flags=re.IGNORECASE), "Global Surface Water", "data_resource_or_product", "Global Surface Water family"),
    (re.compile(r"\bGridded Livestock of the World\b|^GLW\b", flags=re.IGNORECASE), "Gridded Livestock of the World", "data_resource_or_product", "Gridded Livestock of the World family"),
    (re.compile(r"\b(?:Surface Ocean CO2 Atlas|SOCAT)\b", flags=re.IGNORECASE), "SOCAT", "data_resource_or_product", "SOCAT family"),
    (re.compile(r"^GLODAP\b|\bGlobal Ocean Data Analysis Project\b", flags=re.IGNORECASE), "GLODAP", "data_resource_or_product", "GLODAP family"),
    (re.compile(r"^EarthEnv\b", flags=re.IGNORECASE), "EarthEnv", "data_resource_or_product", "EarthEnv family"),
    (re.compile(r"^EarthStat\b", flags=re.IGNORECASE), "EarthStat", "data_resource_or_product", "EarthStat family"),
    (re.compile(r"\bGlobal Land Cover Facility\b|^GLCF\b", flags=re.IGNORECASE), "Global Land Cover Facility", "data_resource_or_product", "Global Land Cover Facility family"),
    (re.compile(r"^CHELSA\b", flags=re.IGNORECASE), "CHELSA", "data_resource_or_product", "CHELSA family"),
    (re.compile(r"\b(?:Global Inventory Modeling and Mapping Studies|Global Inventory Monitoring and Modeling Studies|GIMMS)\b", flags=re.IGNORECASE), "GIMMS", "data_resource_or_product", "GIMMS family"),
    (re.compile(r"^Ensembl\b|^ENSEMBL\b|\bEnsembl(?:Bacteria|Fungi|Plants|Metazoa|Protists|Genomes|Compara)?\b", flags=re.IGNORECASE), "Ensembl", "data_resource_or_product", "Ensembl family"),
    (re.compile(r"^e!Ensembl\b", flags=re.IGNORECASE), "Ensembl", "data_resource_or_product", "Ensembl spelling family"),
    (re.compile(r"^TAIR\b|\bThe Arabidopsis Information Resource\b", flags=re.IGNORECASE), "TAIR", "data_resource_or_product", "TAIR family"),
    (re.compile(r"\b1000 Genomes? Project\b|\b1000 genomes project\b", flags=re.IGNORECASE), "1000 Genomes Project", "data_resource_or_product", "1000 Genomes Project family"),
    (re.compile(r"\b1001 Genomes? Project\b", flags=re.IGNORECASE), "1001 Genomes Project", "data_resource_or_product", "1001 Genomes Project family"),
    (re.compile(r"^HMDB\b|\bHuman Metabolome Database\b", flags=re.IGNORECASE), "HMDB", "data_resource_or_product", "Human Metabolome Database family"),
    (re.compile(r"\bHuman Genome Diversity Project\b|\bHGDP\b", flags=re.IGNORECASE), "Human Genome Diversity Project", "data_resource_or_product", "Human Genome Diversity Project family"),
    (re.compile(r"\bHuman Microbiome Project\b|\bHMP\b", flags=re.IGNORECASE), "Human Microbiome Project", "data_resource_or_product", "Human Microbiome Project family"),
    (re.compile(r"\bHuman Development Index\b|\bHDI\b", flags=re.IGNORECASE), "Human Development Index", "data_resource_or_product", "Human Development Index family"),
    (re.compile(r"\bHuman Footprint\b|\bHuman Influence Index\b", flags=re.IGNORECASE), "Human Footprint Index", "data_resource_or_product", "Human Footprint/Human Influence Index family"),
    (re.compile(r"\b(?:Rice Genome Annotation Project|RGAP|Rice Annotation Project Database)\b", flags=re.IGNORECASE), "Rice Genome Annotation Project", "data_resource_or_product", "Rice Genome Annotation Project family"),
    (re.compile(r"\b(?:3000|3,000) Rice Genomes? Project\b", flags=re.IGNORECASE), "3000 Rice Genomes Project", "data_resource_or_product", "3000 Rice Genomes Project family"),
    (re.compile(r"^MetaCyc\b", flags=re.IGNORECASE), "MetaCyc", "data_resource_or_product", "MetaCyc family"),
    (re.compile(r"^CARD\b|\bComprehensive Antibiotic Resistance Database\b", flags=re.IGNORECASE), "CARD", "data_resource_or_product", "CARD family"),
    (re.compile(r"\bCompTox\b", flags=re.IGNORECASE), "CompTox Chemicals Dashboard", "data_resource_or_product", "CompTox family"),
    (re.compile(r"\b(?:Comparative Toxicogenomics Database|\bCTD\b)", flags=re.IGNORECASE), "Comparative Toxicogenomics Database", "data_resource_or_product", "Comparative Toxicogenomics Database family"),
    (re.compile(r"^ProteomeXchange\b", flags=re.IGNORECASE), "ProteomeXchange", "data_resource_or_product", "ProteomeXchange family"),
    (re.compile(r"\b(?:Joint Genome Institute|\bJGI\b).*\b(?:IMG|Integrated Microbial Genomes)\b", flags=re.IGNORECASE), "JGI IMG", "data_resource_or_product", "JGI IMG family"),
    (re.compile(r"\b(?:Joint Genome Institute|\bJGI\b).*\bMycoCosm\b", flags=re.IGNORECASE), "JGI MycoCosm", "data_resource_or_product", "JGI MycoCosm family"),
    (re.compile(r"\b(?:Joint Genome Institute|\bJGI\b)\b", flags=re.IGNORECASE), "JGI", "data_holder", "DOE Joint Genome Institute holder family"),
    (re.compile(r"\bBirdLife Data Zone\b", flags=re.IGNORECASE), "BirdLife Data Zone", "data_resource_or_product", "BirdLife Data Zone family"),
    (re.compile(r"\bBird\s*Life International\b|\bBirdLife International\b", flags=re.IGNORECASE), "BirdLife International", "data_holder", "BirdLife International holder family"),
    (re.compile(r"\bAtlas of Living Australia\b|\bALA\b", flags=re.IGNORECASE), "Atlas of Living Australia", "data_resource_or_product", "Atlas of Living Australia family"),
    (re.compile(r"\bCalifornia Natural Diversity Database\b|\bCNDDB\b", flags=re.IGNORECASE), "California Natural Diversity Database", "data_resource_or_product", "California Natural Diversity Database family"),
    (re.compile(r"\bSwedish Species Observation System\b", flags=re.IGNORECASE), "Swedish Species Observation System", "data_resource_or_product", "Swedish Species Observation System family"),
    (re.compile(r"\bSwedish Species Gateway\b", flags=re.IGNORECASE), "Swedish Species Gateway", "data_resource_or_product", "Swedish Species Gateway family"),
    (re.compile(r"^Canadensys\b", flags=re.IGNORECASE), "Canadensys", "data_resource_or_product", "Canadensys family"),
    (re.compile(r"\b(?:UNEP[-/ ]?WCMC|World Conservation Monitoring Centre)\b", flags=re.IGNORECASE), "UNEP-WCMC", "data_holder", "UNEP-WCMC holder family"),
    (re.compile(r"\bProtected Planet\b", flags=re.IGNORECASE), "Protected Planet", "data_resource_or_product", "Protected Planet family"),
    (re.compile(r"\bCITES\b.*\bTrade Database\b", flags=re.IGNORECASE), "CITES Trade Database", "data_resource_or_product", "CITES Trade Database family"),
    (re.compile(r"^ICES\s+DATRAS\b|\bICES\b.*\bDATRAS\b", flags=re.IGNORECASE), "ICES DATRAS", "data_resource_or_product", "ICES DATRAS family"),
    (re.compile(r"^ICES\b", flags=re.IGNORECASE), "ICES", "data_holder", "ICES holder family"),
    (re.compile(r"\bALOS\b.*\bPALSAR\b", flags=re.IGNORECASE), "ALOS PALSAR", "data_resource_or_product", "ALOS PALSAR family"),
    (re.compile(r"\b(?:AW3D30|ALOS World 3D(?:-30)?|ALOS Global Digital Surface Model)\b", flags=re.IGNORECASE), "AW3D30", "data_resource_or_product", "ALOS World 3D/AW3D30 family"),
    (re.compile(r"^SPOT\b|\bSPOT\s*(?:[1-7]|VGT|Vegetation)\b", flags=re.IGNORECASE), "SPOT", "data_resource_or_product", "SPOT satellite/product family"),
    (re.compile(r"\bASTER\b.*\bGDEM\b", flags=re.IGNORECASE), "ASTER GDEM", "data_resource_or_product", "ASTER GDEM family"),
    (re.compile(r"\bASTER\b.*\bGED\b", flags=re.IGNORECASE), "ASTER GED", "data_resource_or_product", "ASTER GED family"),
    (re.compile(r"^AVHRR\b|\bAdvanced Very High Resolution Radiometer\b", flags=re.IGNORECASE), "AVHRR", "data_resource_or_product", "AVHRR family"),
    (re.compile(r"^SMHI\b|\bSwedish Meteorological and Hydrological Institute\b", flags=re.IGNORECASE), "SMHI", "data_holder", "SMHI holder family"),
    (re.compile(r"\bKorean Statistical Information Service\b|^KOSIS\b", flags=re.IGNORECASE), "KOSIS", "data_resource_or_product", "Korean Statistical Information Service family"),
    (re.compile(r"\bIBGE\b.*\bSIDRA\b", flags=re.IGNORECASE), "IBGE SIDRA", "data_resource_or_product", "IBGE SIDRA family"),
    (re.compile(r"^OECD\s+iLibrary\b|\bOECD iLibrary\b", flags=re.IGNORECASE), "OECD iLibrary", "data_resource_or_product", "OECD iLibrary family"),
    (re.compile(r"^OECD\b", flags=re.IGNORECASE), "OECD", "data_holder", "OECD holder family"),
    (re.compile(r"\bWorld Register of Marine Species\b|\bWorld Register for Marine Species\b|\bWorld Register of Marines Species\b|\bWORMS\b|\bWoRMS\b", flags=re.IGNORECASE), "WoRMS", "data_resource_or_product", "WoRMS parenthetical/name variant family"),
    (re.compile(r"\bConserved Domain Database\b|\bNCBI\b.*\bCDD\b", flags=re.IGNORECASE), "NCBI Conserved Domain Database", "data_resource_or_product", "NCBI Conserved Domain Database family"),
    (re.compile(r"\bTranscriptome Shotgun Assembly\b|\bNCBI\b.*\bTSA\b", flags=re.IGNORECASE), "NCBI TSA", "data_resource_or_product", "NCBI Transcriptome Shotgun Assembly family"),
    (re.compile(r"\bNASA/IPAC Extragalactic Database\b|\bNASA Extragalactic Database\b", flags=re.IGNORECASE), "NASA/IPAC Extragalactic Database", "data_resource_or_product", "NASA/IPAC Extragalactic Database family"),
    (re.compile(r"\bNational Snow (?:and|&) Ice Data Cent(?:er|re)\b|\bNSIDC\b", flags=re.IGNORECASE), "NSIDC", "data_holder", "NSIDC holder family"),
    (re.compile(r"\bOpen[- ]?Street[- ]?Map\b|\bOSM\b", flags=re.IGNORECASE), "OpenStreetMap", "data_resource_or_product", "OpenStreetMap/OSM family"),
    (re.compile(r"\b(?:KEGG Orthology|KEGG Ortholog|KEGG ontology|\bKO\b)\b", flags=re.IGNORECASE), "KEGG Orthology", "data_resource_or_product", "KEGG Orthology family"),
    (re.compile(r"\b(?:Ag[êe]ncia Nacional de [ÁA]guas|Brazilian National Water Agency|Brazilian Water Agency|National Water Agency)\b.*\bANA\b|\bANA\b.*\bBrazil", flags=re.IGNORECASE), "Brazilian National Water Agency", "data_holder", "Brazilian National Water Agency holder family"),
    (re.compile(r"\b(?:India|Indian) Meteorological Department\b|\bIndian Meteorology Department\b", flags=re.IGNORECASE), "India Meteorological Department", "data_holder", "India Meteorological Department holder family"),
    (re.compile(r"\bIndex of Multiple Deprivation\b|\bEnglish Indices of Multiple Deprivation\b", flags=re.IGNORECASE), "Index of Multiple Deprivation", "data_resource_or_product", "Index of Multiple Deprivation family"),
    (re.compile(r"\bHuman Oral Microbiome Database\b|\bHOMD\b", flags=re.IGNORECASE), "HOMD", "data_resource_or_product", "Human Oral Microbiome Database family"),
    (re.compile(r"\bPermanent Service for (?:the )?Mean Sea Level\b|\bPSMSL\b", flags=re.IGNORECASE), "PSMSL", "data_resource_or_product", "Permanent Service for Mean Sea Level family"),
    (re.compile(r"\bContinuous Plankton Recorder\b|\bCPR\b\s+survey", flags=re.IGNORECASE), "Continuous Plankton Recorder Survey", "data_resource_or_product", "Continuous Plankton Recorder Survey family"),
    (re.compile(r"\bEarth System Grid Federation\b|\bESGF\b", flags=re.IGNORECASE), "ESGF", "data_resource_or_product", "Earth System Grid Federation family"),
    (re.compile(r"\bKorea(?:n)? Meteorolog(?:ical|y) (?:Administration|Agency)\b|\bKMA\b", flags=re.IGNORECASE), "Korea Meteorological Administration", "data_holder", "Korea Meteorological Administration holder family"),
    (re.compile(r"\bAmerican Community Survey\b|\bACS\b.*\bCensus", flags=re.IGNORECASE), "American Community Survey", "data_resource_or_product", "American Community Survey family"),
    (re.compile(r"\bDemographic(?: and)? Health Surveys?\b|\bDemographic and Health Surveys?\b", flags=re.IGNORECASE), "Demographic and Health Surveys", "data_resource_or_product", "Demographic and Health Surveys family"),
    (re.compile(r"\bBangladesh Bureau of Statistics\b", flags=re.IGNORECASE), "Bangladesh Bureau of Statistics", "data_holder", "Bangladesh Bureau of Statistics holder family"),
    (re.compile(r"^\s*EEA\s*$|\bEuropean Environment(?:al)? Agency\b", flags=re.IGNORECASE), "European Environment Agency", "data_holder", "European Environment Agency holder family"),
    (re.compile(r"\bNCBI\b.*\bRefSeq\b|\bReference Sequence(?:s)?\s*\(RefSeq\)|\bRefSeq\b", flags=re.IGNORECASE), "NCBI RefSeq", "data_resource_or_product", "NCBI RefSeq family"),
    (re.compile(r"\bGlobal Land Cover 2000\b|\bGLC[-\s]?2000\b", flags=re.IGNORECASE), "Global Land Cover 2000", "data_resource_or_product", "Global Land Cover 2000 family"),
    (re.compile(r"\bGlobal Lakes? and Wetlands? Database\b|\bGLWD\b", flags=re.IGNORECASE), "Global Lakes and Wetlands Database", "data_resource_or_product", "Global Lakes and Wetlands Database family"),
    (re.compile(r"\b(?:CPC\s+)?Merged Analysis(?: of)? Precipitation\b|\bCMAP\b", flags=re.IGNORECASE), "NOAA CMAP", "data_resource_or_product", "NOAA CPC CMAP precipitation family"),
    (re.compile(r"\bGlobal (?:Surface )?Summary of the Day\b|\bGSOD\b", flags=re.IGNORECASE), "NOAA GSOD", "data_resource_or_product", "NOAA Global Summary of the Day family"),
    (re.compile(r"\bBritish Geological Survey\b|\bBGS\b", flags=re.IGNORECASE), "British Geological Survey", "data_holder", "British Geological Survey holder family"),
    (re.compile(r"\bInternational Energy Agency\b", flags=re.IGNORECASE), "International Energy Agency", "data_holder", "International Energy Agency holder family"),
    (re.compile(r"\bSimple Ocean Data Assimilation\b|\bSODA\b", flags=re.IGNORECASE), "Simple Ocean Data Assimilation", "data_resource_or_product", "Simple Ocean Data Assimilation family"),
    (re.compile(r"\bGlobal Land Survey\b|\bGLS\b", flags=re.IGNORECASE), "Global Land Survey", "data_resource_or_product", "Global Land Survey family"),
    (re.compile(r"\bGolm Metabolome Database\b|\bGOLM Metabolome Database\b", flags=re.IGNORECASE), "Golm Metabolome Database", "data_resource_or_product", "Golm Metabolome Database family"),
    (re.compile(r"\bNOAA\b.*\bGlobal Monitoring Division\b|\bGlobal Monitoring Division\b.*\bNOAA\b", flags=re.IGNORECASE), "NOAA Global Monitoring Division", "data_holder", "NOAA Global Monitoring Division holder family"),
    (re.compile(r"\bNational Wetlands? Inventory\b|\bNWI\b", flags=re.IGNORECASE), "National Wetlands Inventory", "data_resource_or_product", "National Wetlands Inventory family"),
    (re.compile(r"\bWorld Wildlife Fund\b|\bWorld Wide Fund(?: for Nature)?\b|\bWWF\b", flags=re.IGNORECASE), "WWF", "data_holder", "WWF holder family"),
    (re.compile(r"\bTRMM\b.*\bTMPA\b|\bMulti[- ]Satellite Precipitation Analysis\b|\bTMPA\b", flags=re.IGNORECASE), "TRMM TMPA", "data_resource_or_product", "TRMM TMPA family"),
    (re.compile(r"\bNational Meteorological Information Cent(?:er|re)\b.*\bChina\b|\bChina\b.*\bNational Meteorological Information Cent(?:er|re)\b|\bNMIC\b.*\bChina", flags=re.IGNORECASE), "China NMIC", "data_holder", "China National Meteorological Information Center family"),
    (re.compile(r"\bNASA\b.*\bGoddard Space Flight Cent(?:er|re)\b|\bGoddard Space Flight Cent(?:er|re)\b.*\bNASA\b|\bGSFC\b", flags=re.IGNORECASE), "NASA GSFC", "data_holder", "NASA Goddard Space Flight Center holder family"),
    (re.compile(r"\bAutomatic Identification System\b|\bAIS\b\s+data", flags=re.IGNORECASE), "Automatic Identification System", "data_resource_or_product", "Automatic Identification System family"),
    (re.compile(r"\bCoastal Change Analysis Program\b|\bC[- ]CAP\b", flags=re.IGNORECASE), "Coastal Change Analysis Program", "data_resource_or_product", "NOAA Coastal Change Analysis Program family"),
    (re.compile(r"\bCulture Collection of Algae and Protozoa\b|\bCCAP\b", flags=re.IGNORECASE), "Culture Collection of Algae and Protozoa", "data_holder", "Culture Collection of Algae and Protozoa family"),
    (re.compile(r"\bMultivariate (?:ENSO|El Ni(?:n|ñ)o) Index\b|\bMEI\b", flags=re.IGNORECASE), "Multivariate ENSO Index", "data_resource_or_product", "Multivariate ENSO Index family"),
    (re.compile(r"\bOceanic (?:El )?Ni(?:n|ñ)o Index\b|\bOceanic Nino Index\b", flags=re.IGNORECASE), "Oceanic Nino Index", "data_resource_or_product", "Oceanic Nino Index family"),
    (re.compile(r"\bSouthern Oscillation Index\b", flags=re.IGNORECASE), "Southern Oscillation Index", "data_resource_or_product", "Southern Oscillation Index family"),
    (re.compile(r"\bSurvey of India\b", flags=re.IGNORECASE), "Survey of India", "data_holder", "Survey of India holder family"),
    (re.compile(r"\bSwedish National Forest Inventor(?:y|ies)\b", flags=re.IGNORECASE), "Swedish National Forest Inventory", "data_resource_or_product", "Swedish National Forest Inventory family"),
    (re.compile(r"\bFrench National Forest Inventor(?:y|ies)\b", flags=re.IGNORECASE), "French National Forest Inventory", "data_resource_or_product", "French National Forest Inventory family"),
    (re.compile(r"\bSwiss National Forest Inventor(?:y|ies)\b", flags=re.IGNORECASE), "Swiss National Forest Inventory", "data_resource_or_product", "Swiss National Forest Inventory family"),
    (re.compile(r"\bNorwegian National Forest Inventor(?:y|ies)\b", flags=re.IGNORECASE), "Norwegian National Forest Inventory", "data_resource_or_product", "Norwegian National Forest Inventory family"),
    (re.compile(r"\bGerman National Forest Inventor(?:y|ies)\b", flags=re.IGNORECASE), "German National Forest Inventory", "data_resource_or_product", "German National Forest Inventory family"),
    (re.compile(r"\bCanadian National Forest Inventor(?:y|ies)\b", flags=re.IGNORECASE), "Canadian National Forest Inventory", "data_resource_or_product", "Canadian National Forest Inventory family"),
    (re.compile(r"\bFinnish National Forest Inventor(?:y|ies)\b", flags=re.IGNORECASE), "Finnish National Forest Inventory", "data_resource_or_product", "Finnish National Forest Inventory family"),
    (re.compile(r"\bKorean National Forest Inventor(?:y|ies)\b", flags=re.IGNORECASE), "Korean National Forest Inventory", "data_resource_or_product", "Korean National Forest Inventory family"),
    (re.compile(r"\bNational Forest Inventor(?:y|ies)\b", flags=re.IGNORECASE), "National Forest Inventory", "data_resource_or_product", "National Forest Inventory family"),
    (re.compile(r"\bDigital Elevation Models?\s*\(DEM\)|\bDigital Terrain Models?\s*\(DTM\)|^\s*DEM\s*$|^\s*DTM\s*$", flags=re.IGNORECASE), "", "generic_or_ambiguous", "generic elevation/terrain model label"),
    (re.compile(r"\bGlobal Digital Elevation Model\b|\bGDEM\b", flags=re.IGNORECASE), "Global Digital Elevation Model", "data_resource_or_product", "Global Digital Elevation Model family"),
    (re.compile(r"\bIUCN\b.*\bred list\b|\bred list of the international union for (?:the )?conservation of nature\b", flags=re.IGNORECASE), "IUCN Red List", "data_resource_or_product", "IUCN Red List family"),
    (re.compile(r"^(?!.*reprojection tool).*\bMODIS\b", flags=re.IGNORECASE), "MODIS", "data_resource_or_product", "MODIS product/family variant"),
    (re.compile(r"\bLandsat\b", flags=re.IGNORECASE), "Landsat", "data_resource_or_product", "Landsat product/family variant"),
    (re.compile(r"\b(?:SRTM|shuttle radar topography missions?)\b", flags=re.IGNORECASE), "SRTM", "data_resource_or_product", "SRTM product/family variant"),
    (re.compile(r"\bSeaW?iFS\b", flags=re.IGNORECASE), "SeaWiFS", "data_resource_or_product", "SeaWiFS product/family variant"),
    (re.compile(r"\bNASA[-\s]?POWER\b|\bPrediction of Worldwide Energy Resources\b|\bpower\s+larc\s+nasa\b", flags=re.IGNORECASE), "NASA POWER", "data_resource_or_product", "NASA POWER product family"),
    (re.compile(r"\b(?:LP\s*DAAC|LPDAAC)\b", flags=re.IGNORECASE), "NASA LP DAAC", "data_resource_or_product", "NASA LP DAAC family"),
    (re.compile(r"\b(?:OBPG|ocean biology processing group)\b", flags=re.IGNORECASE), "NASA OBPG", "data_resource_or_product", "NASA Ocean Biology Processing Group family"),
    (re.compile(r"\bNASA\b.*\bMEaSUREs\b|\bMaking Earth System Data Records\b", flags=re.IGNORECASE), "NASA MEaSUREs", "data_resource_or_product", "NASA MEaSUREs family"),
    (re.compile(r"\b(?:ncep\s*/\s*ncar|ncar\s*/\s*ncep|ncep[-\s]+ncar|ncar[-\s]+ncep)\b", flags=re.IGNORECASE), "NCEP/NCAR Reanalysis", "data_resource_or_product", "NCEP/NCAR reanalysis family"),
    (re.compile(r"\bNCEP\b.*\bNational Cent(?:er|re) for Atmospheric Research\b.*\breanalysis\b", flags=re.IGNORECASE), "NCEP/NCAR Reanalysis", "data_resource_or_product", "NCEP/NCAR reanalysis family"),
    (re.compile(r"\bNCEP\s*/?\s*DOE\b", flags=re.IGNORECASE), "NCEP/DOE Reanalysis 2", "data_resource_or_product", "NCEP/DOE Reanalysis family"),
    (re.compile(r"\b(?:NCEP\s+)?(?:climate forecast system reanalysis|CFSR)\b", flags=re.IGNORECASE), "NCEP CFSR", "data_resource_or_product", "NCEP CFSR family"),
    (re.compile(r"\bNational Cent(?:er|re) for Atmospheric Research\b|\bClimate Analysis Section\b.*\bNCAR\b|^\s*NCAR\s*$", flags=re.IGNORECASE), "NCAR", "data_holder", "NCAR holder family"),
    (re.compile(r"\b(?:ECMWF\b.*\b(?:interim|ERA[-\s]?interim)|ERA[-\s]?interim|European Reanalysis Interim)\b", flags=re.IGNORECASE), "ERA-Interim", "data_resource_or_product", "ECMWF ERA-Interim family"),
    (re.compile(r"\b(?:ECMWF\b.*\bERA[-\s]?5|ERA[-\s]?5)\b", flags=re.IGNORECASE), "ERA5", "data_resource_or_product", "ECMWF ERA5 family"),
    (re.compile(r"\b(?:ECMWF\b.*\bERA[-\s]?40|ERA[-\s]?40|ERA40)\b", flags=re.IGNORECASE), "ERA-40", "data_resource_or_product", "ECMWF ERA-40 family"),
    (re.compile(r"\b(?:ORAS5|ocean reanalysis system 5)\b", flags=re.IGNORECASE), "ECMWF ORAS5", "data_resource_or_product", "ECMWF ORAS5 family"),
    (re.compile(r"\bECMWF\b.*\b(?:HRES|integrated forecasting system|IFS)\b", flags=re.IGNORECASE), "ECMWF IFS", "data_resource_or_product", "ECMWF IFS/HRES family"),
    (re.compile(r"\b(?:FAOSTAT)\b", flags=re.IGNORECASE), "FAOSTAT", "data_resource_or_product", "FAOSTAT family"),
    (re.compile(r"\bGlobal Administrative Unit Layers?\b|\bFAO\b.*\bGAUL\b|^GAUL\b", flags=re.IGNORECASE), "FAO GAUL", "data_resource_or_product", "FAO GAUL family"),
    (re.compile(r"\bFAO\b.*\b(?:forest resources assessment|FRA)\b", flags=re.IGNORECASE), "FAO FRA", "data_resource_or_product", "FAO Forest Resources Assessment family"),
    (re.compile(r"\bWorld Development Indicators\b|\bWorld Bank\b.*\bWDI\b", flags=re.IGNORECASE), "World Development Indicators", "data_resource_or_product", "World Development Indicators family"),
    (re.compile(r"\b(?:Worldwide Governance Indicators|World Governance Indicators)\b|\bWorld Bank\b.*\bWGI\b", flags=re.IGNORECASE), "Worldwide Governance Indicators", "data_resource_or_product", "Worldwide Governance Indicators family"),
    (re.compile(r"\bWorld Bank Open Data\b|data\.worldbank\.org", flags=re.IGNORECASE), "World Bank Open Data", "data_resource_or_product", "World Bank Open Data family"),
    (re.compile(r"\b(?:ERSST|extended reconstructed sea surface temperature)\b", flags=re.IGNORECASE), "NOAA ERSST", "data_resource_or_product", "NOAA ERSST family"),
    (re.compile(r"\b(?:OISST|optimum interpolation sea surface temperature|OI\s+SST)\b", flags=re.IGNORECASE), "NOAA OISST", "data_resource_or_product", "NOAA OISST family"),
    (re.compile(r"\bNOAA\b.*\bCoral Reef Watch\b", flags=re.IGNORECASE), "NOAA Coral Reef Watch", "data_resource_or_product", "NOAA Coral Reef Watch family"),
    (re.compile(r"\bNOAA\b.*\bCoastWatch\s*/?\s*OceanWatch\b", flags=re.IGNORECASE), "NOAA CoastWatch/OceanWatch", "data_resource_or_product", "NOAA CoastWatch/OceanWatch family"),
    (re.compile(r"\bNOAA\b.*\bNational Weather Service\b|^National Weather Service\b", flags=re.IGNORECASE), "NOAA National Weather Service", "data_holder", "NOAA National Weather Service holder family"),
    (re.compile(r"\bNOAA\b.*\bOffice for Coastal Management\b", flags=re.IGNORECASE), "NOAA Office for Coastal Management", "data_holder", "NOAA Office for Coastal Management holder family"),
    (re.compile(r"^NOAA[-\s]?NOS\b|^NOS\b", flags=re.IGNORECASE), "NOAA NOS", "data_holder", "NOAA National Ocean Service holder family"),
    (re.compile(r"\bBreeding Bird Survey\b", flags=re.IGNORECASE), "USGS Breeding Bird Survey", "data_resource_or_product", "USGS Breeding Bird Survey family"),
    (re.compile(r"\bNAWQA\b|\bNational Water-Quality Assessment\b", flags=re.IGNORECASE), "USGS NAWQA", "data_resource_or_product", "USGS NAWQA family"),
    (re.compile(r"\bUSGS\b.*\bNAS\b|\bNonindigenous Aquatic Species\b", flags=re.IGNORECASE), "USGS NAS", "data_resource_or_product", "USGS Nonindigenous Aquatic Species family"),
    (re.compile(r"\bNational Land Cover Database\b|\bNLCD\b", flags=re.IGNORECASE), "National Land Cover Database", "data_resource_or_product", "National Land Cover Database family"),
    (re.compile(r"\b(?:BIOSIS|Biological Abstracts)\b", flags=re.IGNORECASE), "BIOSIS Biological Abstracts", "bibliographic_database", "BIOSIS bibliographic database family"),
    (re.compile(r"\bWeb of Science\b", flags=re.IGNORECASE), "Web of Science", "bibliographic_database", "Web of Science bibliographic database family"),
    (re.compile(r"\bPubMed\b", flags=re.IGNORECASE), "PubMed", "bibliographic_database", "PubMed bibliographic database family"),
    (re.compile(r"\bScopus\b", flags=re.IGNORECASE), "Scopus", "bibliographic_database", "Scopus bibliographic database family"),
    (re.compile(r"\bGoogle Scholar\b", flags=re.IGNORECASE), "Google Scholar", "bibliographic_database", "Google Scholar bibliographic database family"),
    (re.compile(r"\bCAB(?:I)?\s+Abstracts\b", flags=re.IGNORECASE), "CAB Abstracts", "bibliographic_database", "CAB Abstracts bibliographic database family"),
    (re.compile(r"\bZoological Record\b", flags=re.IGNORECASE), "Zoological Record", "bibliographic_database", "Zoological Record bibliographic database family"),
    (re.compile(r"^Dimensions\b|dimensions\.ai|app\.dimensions\.ai", flags=re.IGNORECASE), "Dimensions", "bibliographic_database", "Dimensions bibliographic database family"),
    (re.compile(r"^(?:ncep\s*/\s*ncar|ncep[-\s]+ncar)\b", flags=re.IGNORECASE), "NCEP/NCAR Reanalysis", "data_resource_or_product", "NCEP/NCAR reanalysis family"),
    (re.compile(r"^(?:ecmwf\s+)?era[-\s]?interim\b", flags=re.IGNORECASE), "ERA-Interim", "data_resource_or_product", "ERA-Interim spelling/product family"),
    (re.compile(r"^(?:ecmwf\s+)?era[-\s]?40\b", flags=re.IGNORECASE), "ERA-40", "data_resource_or_product", "ERA-40 spelling/product family"),
    (re.compile(r"^(?:ecmwf\s+)?era[-\s]?5\b", flags=re.IGNORECASE), "ERA5", "data_resource_or_product", "ERA5 spelling/product family"),
    (re.compile(r"^(?:cru\s*[- ]?ts\s*\d*|(?:climate|climatic) research unit.*\bts\s*\d)", flags=re.IGNORECASE), "CRU TS", "data_resource_or_product", "CRU TS version family"),
    (re.compile(r"^(?:climate|climatic) research unit\b", flags=re.IGNORECASE), "CRU", "data_holder", "Climate Research Unit holder family"),
    (re.compile(r"^world ocean database\b|^noaa\s+world ocean database\b", flags=re.IGNORECASE), "World Ocean Database", "data_resource_or_product", "World Ocean Database version family"),
    (re.compile(r"^(?:esa\s*[-/]?\s*cci|european space agency.*climate change initiative)\b", flags=re.IGNORECASE), "ESA CCI", "data_resource_or_product", "ESA Climate Change Initiative family"),
    (re.compile(r"^ncbi\s+taxonomy\b", flags=re.IGNORECASE), "NCBI Taxonomy", "data_resource_or_product", "NCBI Taxonomy family"),
    (re.compile(r"^ncbi\s+genome\b", flags=re.IGNORECASE), "NCBI Genome", "data_resource_or_product", "NCBI Genome family"),
    (re.compile(r"^ncbi\s+(?:non[-\s]?redundant|nr)\b", flags=re.IGNORECASE), "NCBI nr", "data_resource_or_product", "NCBI nonredundant sequence family"),
    (re.compile(r"^(?:gene\s*bank|genebank)\s+accession\b", flags=re.IGNORECASE), "GenBank", "data_resource_or_product", "GenBank accession spelling family"),
    (re.compile(r"^whole genome shotgun\b", flags=re.IGNORECASE), "NCBI WGS", "data_resource_or_product", "Whole Genome Shotgun family"),
    (re.compile(r"^tiger\s*/?\s*line\b|^\d{4}\s+(?:u\.?s\.?\s+census\s+)?tiger\s*/?\s*line\b", flags=re.IGNORECASE), "TIGER/Line Shapefiles", "data_resource_or_product", "TIGER/Line shapefile family"),
    (re.compile(r"^(?:us\s+geological survey|united states geological survey|usgs).*earth\s*explorer\b", flags=re.IGNORECASE), "USGS EarthExplorer", "data_resource_or_product", "USGS EarthExplorer family"),
    (re.compile(r"^(?:us\s+geological survey|united states geological survey|usgs).*national water information system\b|^nwis\b", flags=re.IGNORECASE), "USGS NWIS", "data_resource_or_product", "USGS National Water Information System family"),
    (re.compile(r"^(?:us\s+geological survey|united states geological survey|usgs).*national elevation dataset\b|^national elevation dataset\b", flags=re.IGNORECASE), "National Elevation Dataset", "data_resource_or_product", "USGS National Elevation Dataset family"),
    (re.compile(r"^(?:us\s+geological survey|united states geological survey|usgs).*global visualization viewer\b|^usgs\s+glovis\b", flags=re.IGNORECASE), "USGS GloVis", "data_resource_or_product", "USGS GloVis family"),
    (re.compile(r"^forest inventory (?:and|&) analysis\b|^forest inventory analysis\b", flags=re.IGNORECASE), "Forest Inventory and Analysis", "data_resource_or_product", "Forest Inventory and Analysis family"),
    (re.compile(r"^(?:noaa\s+)?(?:national climatic data center|national climate data center|ncdc)\b", flags=re.IGNORECASE), "NOAA NCDC", "data_holder", "NOAA NCDC holder family"),
    (re.compile(r"^(?:noaa\s+)?(?:national centers? for environmental information|ncei)\b", flags=re.IGNORECASE), "NOAA NCEI", "data_holder", "NOAA NCEI holder family"),
    (re.compile(r"^(?:noaa\s+)?(?:climate prediction cent(?:er|re)|cpc)\b", flags=re.IGNORECASE), "NOAA CPC", "data_holder", "NOAA Climate Prediction Center holder family"),
    (re.compile(r"^noaa\s*[-/]?\s*esrl\b|^esrl\b", flags=re.IGNORECASE), "NOAA ESRL", "data_holder", "NOAA ESRL holder family"),
    (re.compile(r"^(?:deutscher wetterdienst|german weather service|german meteorological service|dwd)\b", flags=re.IGNORECASE), "DWD", "data_holder", "German weather service holder family"),
    (re.compile(r"^(?:us\s+census bureau|us\s+census)\b", flags=re.IGNORECASE), "US Census Bureau", "data_holder", "US Census Bureau holder family"),
    (re.compile(r"^china meteorological administration\b|^cma\b", flags=re.IGNORECASE), "CMA", "data_holder", "China Meteorological Administration holder family"),
    (re.compile(r"^data center for resources and environmental sciences.*\bresdc\b|^resdc\b", flags=re.IGNORECASE), "RESDC", "data_holder", "RESDC holder family"),
    (re.compile(r"^(?:cluster|clusters) of orthologous groups\b", flags=re.IGNORECASE), "COG", "data_resource_or_product", "COG family"),
    (re.compile(r"^ucsc\s+genome(?: browser)?\b", flags=re.IGNORECASE), "UCSC Genome Browser", "data_resource_or_product", "UCSC Genome Browser family"),
    (re.compile(r"^global runoff data cent(?:re|er)\b|^grdc\b", flags=re.IGNORECASE), "GRDC", "data_holder", "Global Runoff Data Centre family"),
    (re.compile(r"^germplasm resources information network\b|^grin\b", flags=re.IGNORECASE), "GRIN", "data_resource_or_product", "GRIN family"),
]

DOI_REPOSITORY_PATTERNS = [
    (re.compile(r"dryad", flags=re.IGNORECASE), "Dryad"),
    (re.compile(r"figshare|10\.6084/m9\.figshare", flags=re.IGNORECASE), "Figshare"),
    (re.compile(r"pangaea|10\.1594/pangaea", flags=re.IGNORECASE), "PANGAEA"),
    (re.compile(r"zenodo|10\.5281/zenodo", flags=re.IGNORECASE), "Zenodo"),
    (re.compile(r"10\.5067/MODIS", flags=re.IGNORECASE), "MODIS"),
]
GENERIC_DOI_RE = re.compile(r"(?:\b(?:https?://)?(?:dx\.)?doi\.org\s*/\s*10\.|\bdoi\s*[:/]\s*10\.|\b10\.\d{4,9}/)", flags=re.IGNORECASE)

TECHNICAL_INFRASTRUCTURE_HOLDER_PATTERNS = [
    (re.compile(r"^sentinel[- ]?1\b", flags=re.IGNORECASE), "ESA", "Sentinel-1 operating institution"),
    (re.compile(r"^sentinel[- ]?2\b", flags=re.IGNORECASE), "ESA", "Sentinel-2 operating institution"),
]


RESOURCE_TO_HOLDER = {
    "GenBank": "NCBI",
    "NCBI Nucleotide": "NCBI",
    "NCBI SRA": "NCBI",
    "NCBI GEO": "NCBI",
    "NCBI RefSeq": "NCBI",
    "NCBI BioSample": "NCBI",
    "NCBI BioProject": "NCBI",
    "NCBI Taxonomy": "NCBI",
    "NCBI Genome": "NCBI",
    "NCBI nr": "NCBI",
    "NCBI Assembly": "NCBI",
    "NCBI WGS": "NCBI",
    "dbGaP": "NCBI",
    "ArrayExpress": "EMBL-EBI",
    "IUCN Red List": "IUCN",
    "GBIF": "GBIF",
    "World Database on Protected Areas": "UNEP-WCMC/IUCN",
    "World Database on Other Effective Area-Based Conservation Measures": "UNEP-WCMC/IUCN",
    "World Database of Key Biodiversity Areas": "KBA Partnership",
    "WWF Ecoregions": "WWF",
    "Climate-Data.org": "Climate-Data.org",
    "NCAR Climate Data Guide": "NCAR",
    "KNMI Climate Explorer": "KNMI",
    "NCAR Climate Analysis Section": "NCAR",
    "DigitalGlobe": "Maxar",
    "ArcGIS Online World Imagery": "Esri",
    "Google Earth": "Google",
    "Google Earth Engine": "Google",
    "Google Maps": "Google",
    "Google Scholar": "Google",
    "Google Trends": "Google",
    "Google Books": "Google",
    "Google Images": "Google",
    "Google Search": "Google",
    "ECA&D": "KNMI",
    "ESDAC": "European Commission JRC",
    "EUNIS": "European Environment Agency",
    "E-PRTR": "European Environment Agency",
    "European Vegetation Archive": "European Vegetation Survey",
    "EFFIS": "European Commission JRC",
    "European Soil Database": "European Commission JRC",
    "European Register of Marine Species": "VLIZ",
    "European Diatom Database": "European Diatom Database",
    "CNKI": "CNKI",
    "China Meteorological Forcing Dataset": "China Meteorological Administration",
    "China Meteorological Data Service Center": "China Meteorological Administration",
    "China National Environmental Monitoring Center": "China National Environmental Monitoring Center",
    "China Statistical Yearbook": "National Bureau of Statistics of China",
    "ForestPlots.net": "ForestPlots.net",
    "Natural Earth": "Natural Earth",
    "Natura 2000": "European Environment Agency",
    "NatureServe Explorer": "NatureServe",
    "International Soil Moisture Network": "International Soil Moisture Network",
    "ICOADS": "NOAA",
    "ISCCP": "NASA",
    "International Tree-Ring Data Bank": "NOAA NCEI",
    "IBTrACS": "NOAA",
    "International Waterbird Census": "Wetlands International",
    "IMF World Economic Outlook": "International Monetary Fund",
    "Copernicus Climate Change Service": "Copernicus/ECMWF",
    "Copernicus Global Land Service": "Copernicus",
    "Copernicus Land Monitoring Service": "Copernicus/European Environment Agency",
    "Copernicus Open Access Hub": "Copernicus/ESA",
    "Copernicus DIAS": "Copernicus",
    "NASA Ocean Color Web": "NASA",
    "Ocean Colour CCI": "ESA",
    "OceanSITES": "OceanSITES",
    "Ocean Tracking Network": "Ocean Tracking Network",
    "USDA PLANTS Database": "USDA NRCS",
    "Plants of the World Online": "Royal Botanic Gardens Kew",
    "Marine Reservoir Correction Database": "Marine Reservoir Database",
    "MarineTraffic": "MarineTraffic",
    "Marine Regions": "Marine Regions",
    "Marine Geoscience Data System": "Lamont-Doherty Earth Observatory",
    "Marine Recreational Information Program": "NOAA Fisheries",
    "Flora do Brasil": "Jardim Botanico do Rio de Janeiro",
    "Flora of North America": "Flora of North America Association",
    "Flora of China": "Flora of China Project",
    "FloraWeb": "Bundesamt fur Naturschutz",
    "UniProt": "UniProt Consortium",
    "UniRef": "UniProt Consortium",
    "Swiss-Prot": "UniProt Consortium",
    "TrEMBL": "UniProt Consortium",
    "UniProt-GOA": "UniProt Consortium",
    "Global Human Settlement Layer": "European Commission JRC",
    "Global Invasive Species Database": "IUCN ISSG",
    "Global Names Index": "Catalogue of Life",
    "Global Carbon Project": "Global Carbon Project",
    "Global Fishing Watch": "Global Fishing Watch",
    "Global Plants": "JSTOR",
    "Global Land Cover Characterization": "USGS",
    "Global Agro-Ecological Zones": "FAO/IIASA",
    "GLC-SHARE": "FAO",
    "WorldPop": "WorldPop",
    "World Population Prospects": "United Nations Population Division",
    "Harmonized World Soil Database": "FAO/IIASA",
    "World Ocean Atlas": "NOAA NCEI",
    "National Biodiversity Network": "National Biodiversity Network",
    "National Ecological Observatory Network": "NEON",
    "National Atmospheric Deposition Program": "National Atmospheric Deposition Program",
    "NHDPlus": "USGS/EPA",
    "National Hydrography Dataset": "USGS",
    "UNEP Regional Seas Programme": "UNEP",
    "UNCCD PRAIS": "UNCCD",
    "UN SDG Indicators Database": "UN Statistics Division",
    "Global SDG Indicator Framework": "UN Statistics Division",
    "System of Environmental-Economic Accounting": "United Nations",
    "SoilGrids": "ISRIC",
    "RiverATLAS": "HydroSHEDS",
    "Restor": "Restor",
    "IUCN Red List of Ecosystems": "IUCN",
    "Protected Area Representativeness and Connectedness": "",
    "Project Information Sharing Framework": "",
    "PREDICTS": "PREDICTS Project",
    "Pesticide Properties DataBase": "University of Hertfordshire",
    "OBIS": "UNESCO-IOC",
    "MODIS Vegetation Continuous Fields": "NASA",
    "Living Planet Database": "ZSL/WWF",
    "Land-Use Harmonization": "LUH Project",
    "International Coral Reef Initiative": "ICRI",
    "Global Register of Introduced and Invasive Species": "IUCN ISSG",
    "Global Pesticide Grids": "",
    "Global Ecosystems Atlas": "",
    "Global Database on Protected Area Management Effectiveness": "UNEP-WCMC/IUCN",
    "Global Biodiversity Change Indicators": "",
    "Framework for Ecosystem Restoration Monitoring": "FAO",
    "Forest Landscape Integrity Index": "",
    "ESA Land Cover": "ESA",
    "Ecosystem Intactness Index": "",
    "CSIRO Biodiversity Indicators Explorer": "CSIRO",
    "CSIRO Data Access Portal": "CSIRO",
    "CROPGRIDS": "",
    "CBD Global CHM": "Convention on Biological Diversity",
    "Biodiversity Intactness Index": "Natural History Museum",
    "Biodiversity Indicators Partnership Dashboard": "Biodiversity Indicators Partnership",
    "Biodiversity Habitat Index": "",
    "Bioclimatic Ecosystem Resilience Index": "",
    "Alien Species First Record Database": "",
    "MODIS": "NASA",
    "Landsat": "USGS/NASA",
    "SRTM": "NASA",
    "TRMM": "NASA/JAXA",
    "World Ocean Atlas": "NOAA/NODC",
    "World Ocean Database": "NOAA/NCEI",
    "ESA CCI": "ESA",
    "Forest Inventory and Analysis": "USDA Forest Service",
    "TIGER/Line Shapefiles": "US Census Bureau",
    "USGS NWIS": "USGS",
    "National Elevation Dataset": "USGS",
    "US Census Bureau": "US Census Bureau",
    "NOAA NCDC": "NOAA",
    "NOAA NCEI": "NOAA",
    "NOAA CPC": "NOAA",
    "NOAA ESRL": "NOAA",
    "DWD": "DWD",
    "CMA": "CMA",
    "RESDC": "Chinese Academy of Sciences",
    "UCSC Genome Browser": "University of California, Santa Cruz",
    "GRDC": "Federal Institute of Hydrology (BfG)",
    "GRIN": "USDA",
    "ERA-Interim": "ECMWF",
    "ERA5": "ECMWF",
    "NCEP Reanalysis": "NOAA/NCEP",
    "NCEP/NCAR Reanalysis": "NOAA/NCEP and NCAR",
    "Google Earth": "Google",
    "Google Earth Pro": "Google",
    "Google Maps": "Google",
    "ESA": "ESA",
    "Web of Science": "Clarivate",
    "Scopus": "Elsevier",
    "Google Scholar": "Google",
    "PubMed": "NCBI",
    "BIOSIS": "Clarivate",
    "BIOSIS Biological Abstracts": "Clarivate",
    "BIOSIS Citation Index": "Clarivate",
    "CAB Abstracts": "CABI",
    "Zoological Record": "Clarivate",
    "Dimensions": "Digital Science",
    "GADM": "GADM",
    "CORINE Land Cover": "European Environment Agency",
    "Copernicus Marine Service": "Copernicus/European Commission",
    "Copernicus Atmosphere Monitoring Service": "Copernicus/ECMWF",
    "Copernicus Climate Data Store": "Copernicus/ECMWF",
    "Copernicus EMS": "Copernicus/European Commission",
    "ESA GlobCover": "ESA",
    "ESA SMOS": "ESA",
    "NASA Earthdata/EOSDIS": "NASA",
    "NASA JPL": "NASA",
    "NASA GES DISC": "NASA",
    "NOAA GDAS": "NOAA/NCEP",
    "NOAA GFS": "NOAA/NCEP",
    "NOAA FNL": "NOAA/NCEP",
    "NOAA GODAS": "NOAA/NCEP",
    "NOAA NARR": "NOAA/NCEP",
    "NOAA NGDC": "NOAA",
    "NOAA NODC": "NOAA",
    "NOAA NDBC": "NOAA",
    "NOAA Global Monitoring Laboratory": "NOAA",
    "NOAA ETOPO": "NOAA",
    "USGS ScienceBase": "USGS",
    "USGS EROS": "USGS",
    "USGS GMTED2010": "USGS",
    "USGS Bird Banding Laboratory": "USGS",
    "US EPA": "US EPA",
    "EPA AQS": "US EPA",
    "EPA ECOTOX": "US EPA",
    "EPA National Emissions Inventory": "US EPA",
    "EPA IRIS": "US EPA",
    "EPA Ecoregions": "US EPA",
    "USDA NRCS": "USDA",
    "USDA NASS": "USDA",
    "USDA ERS": "USDA",
    "USDA SSURGO": "USDA NRCS",
    "USDA Cropland Data Layer": "USDA NASS",
    "USDA NAIP": "USDA",
    "USDA National Nutrient Database": "USDA",
    "FAO AQUASTAT": "FAO",
    "FAO FishStat": "FAO",
    "FAO Food Balance Sheets": "FAO",
    "FAO GeoNetwork": "FAO",
    "FAO Soil Map of the World": "FAO",
    "NIST Standard Reference Materials": "NIST",
    "NIST Mass Spectral Library": "NIST",
    "UK Land Cover Map": "UK Centre for Ecology & Hydrology",
    "UK Countryside Survey": "UK Centre for Ecology & Hydrology",
    "UK Ordnance Survey MasterMap": "Ordnance Survey",
    "UK National River Flow Archive": "UK Centre for Ecology & Hydrology",
    "UK Met Office Hadley Centre": "Met Office",
    "Environment Canada": "Environment and Climate Change Canada",
    "Environment Agency": "Environment Agency",
    "ECHA": "European Chemicals Agency",
    "European Commission JRC": "European Commission",
    "EMODnet": "European Commission",
    "European Fire Database": "European Commission JRC",
    "EGA": "EMBL-EBI/CRG",
    "EMEP": "EMEP",
    "WorldPop": "University of Southampton",
    "GPW": "CIESIN",
    "HYDE": "PBL Netherlands Environmental Assessment Agency",
    "GHCN": "NOAA",
    "GLDAS": "NASA",
    "NLDAS": "NASA/NOAA",
    "Global Surface Water": "European Commission JRC",
    "Gridded Livestock of the World": "FAO",
    "SOCAT": "SOCAT",
    "GLODAP": "GLODAP",
    "EarthEnv": "EarthEnv",
    "EarthStat": "EarthStat",
    "Global Land Cover Facility": "University of Maryland",
    "CHELSA": "WSL",
    "GIMMS": "NASA GIMMS",
    "NARR": "NOAA/NCEP",
    "NCEP FNL": "NOAA/NCEP",
    "NCEP GFS": "NOAA/NCEP",
    "AERONET": "NASA",
    "Ensembl": "EMBL-EBI",
    "Ensembl Genomes": "EMBL-EBI",
    "TAIR": "TAIR",
    "1000 Genomes Project": "International Genome Sample Resource",
    "1001 Genomes Project": "1001 Genomes Consortium",
    "HMDB": "HMDB",
    "Human Genome Diversity Project": "HGDP",
    "Human Microbiome Project": "NIH",
    "Human Development Index": "UNDP",
    "Human Footprint Index": "SEDAC/WCS",
    "Rice Genome Annotation Project": "MSU",
    "3000 Rice Genomes Project": "International Rice Research Institute",
    "MetaCyc": "SRI International",
    "CARD": "McMaster University",
    "CompTox Chemicals Dashboard": "US EPA",
    "Comparative Toxicogenomics Database": "CTD",
    "ProteomeXchange": "ProteomeXchange Consortium",
    "JGI IMG": "DOE Joint Genome Institute",
    "JGI MycoCosm": "DOE Joint Genome Institute",
    "JGI": "DOE Joint Genome Institute",
    "BirdLife International": "BirdLife International",
    "BirdLife Data Zone": "BirdLife International",
    "Atlas of Living Australia": "Atlas of Living Australia",
    "California Natural Diversity Database": "California Department of Fish and Wildlife",
    "Swedish Species Observation System": "Swedish Species Information Centre",
    "Swedish Species Gateway": "Swedish Species Information Centre",
    "Canadensys": "Canadensys",
    "UNEP-WCMC": "UNEP-WCMC",
    "Protected Planet": "UNEP-WCMC/IUCN",
    "CITES Trade Database": "UNEP-WCMC",
    "ICES": "ICES",
    "ICES DATRAS": "ICES",
    "ALOS PALSAR": "JAXA",
    "AW3D30": "JAXA",
    "SPOT": "CNES/Airbus",
    "ASTER GDEM": "NASA/METI",
    "ASTER GED": "NASA/METI",
    "AVHRR": "NOAA",
    "SMHI": "SMHI",
    "KOSIS": "Statistics Korea",
    "IBGE SIDRA": "IBGE",
    "OECD": "OECD",
    "OECD iLibrary": "OECD",
    "NCBI Conserved Domain Database": "NCBI",
    "NCBI TSA": "NCBI",
    "NASA/IPAC Extragalactic Database": "NASA/IPAC",
    "NSIDC": "NSIDC",
    "OpenStreetMap": "OpenStreetMap Foundation",
    "KEGG Orthology": "Kanehisa Laboratories",
    "Brazilian National Water Agency": "ANA",
    "India Meteorological Department": "India Meteorological Department",
    "Index of Multiple Deprivation": "UK Ministry of Housing, Communities and Local Government",
    "NASA GES DISC": "NASA",
    "HOMD": "HOMD",
    "PSMSL": "National Oceanography Centre",
    "Continuous Plankton Recorder Survey": "Marine Biological Association",
    "ESGF": "Earth System Grid Federation",
    "Korea Meteorological Administration": "KMA",
    "American Community Survey": "US Census Bureau",
    "Demographic and Health Surveys": "DHS Program",
    "Bangladesh Bureau of Statistics": "Bangladesh Bureau of Statistics",
    "European Environment Agency": "European Environment Agency",
    "Global Land Cover 2000": "European Commission JRC",
    "Global Lakes and Wetlands Database": "WWF/CIESIN",
    "NOAA CMAP": "NOAA CPC",
    "NOAA GSOD": "NOAA/NCEI",
    "British Geological Survey": "British Geological Survey",
    "International Energy Agency": "IEA",
    "Simple Ocean Data Assimilation": "SODA Consortium",
    "Global Land Survey": "USGS/NASA",
    "Golm Metabolome Database": "Max Planck Institute",
    "NOAA Global Monitoring Division": "NOAA",
    "National Wetlands Inventory": "US Fish and Wildlife Service",
    "WWF": "WWF",
    "TRMM TMPA": "NASA/JAXA",
    "China NMIC": "China Meteorological Administration",
    "NASA GSFC": "NASA",
    "Automatic Identification System": "AIS",
    "Coastal Change Analysis Program": "NOAA",
    "Culture Collection of Algae and Protozoa": "CCAP",
    "Multivariate ENSO Index": "NOAA",
    "Oceanic Nino Index": "NOAA CPC",
    "Southern Oscillation Index": "NOAA CPC",
    "Survey of India": "Survey of India",
    "NCAR": "NCAR",
    "Global Digital Elevation Model": "NASA/METI",

    "KEGG": "Kanehisa Laboratories",
    "Pfam": "EMBL-EBI",
    "UniProt": "UniProt Consortium",
    "SILVA": "SILVA Consortium",
    "Greengenes": "Second Genome/LBNL",
    "RDP": "Michigan State University",
    "RDP Classifier": "Michigan State University",
    "BOLD": "Centre for Biodiversity Genomics",
    "MG-RAST": "Argonne National Laboratory",
    "European Nucleotide Archive": "EMBL-EBI",
    "EMBL Nucleotide Sequence Database": "EMBL-EBI",
    "DDBJ": "National Institute of Genetics, Japan",
    "INSDC": "DDBJ/EMBL-EBI/NCBI",
    "Swiss-Prot": "UniProt Consortium",
    "Protein Data Bank": "wwPDB",
    "Gene Ontology": "Gene Ontology Consortium",
    "COG": "NCBI",
    "UNITE": "UNITE Community",
    "Dryad": "Dryad",
    "Figshare": "Figshare",
    "Zenodo": "CERN/OpenAIRE",
    "PANGAEA": "AWI/MARUM",
    "FishBase": "FishBase Consortium",
    "TreeBASE": "TreeBASE",
    "MycoBank": "Westerdijk Fungal Biodiversity Institute",
    "WoRMS": "Flanders Marine Institute",
    "FAOSTAT": "FAO",
    "FAO": "FAO",
    "WorldClim": "WorldClim",
    "CMIP5": "PCMDI/ESGF",
    "CMIP6": "PCMDI/ESGF",
    "CRU": "University of East Anglia Climatic Research Unit",
    "CRU TS": "University of East Anglia Climatic Research Unit",
    "HadCRUT": "Met Office Hadley Centre/CRU",
    "HadISST": "Met Office Hadley Centre",
    "ERA-40": "ECMWF",
    "GPCP": "NASA/NOAA",
    "GRACE": "NASA/DLR",
    "ASTER": "NASA/METI",
    "ASTER GDEM": "NASA/METI",
    "AVHRR": "NOAA",
    "SeaWiFS": "NASA",
    "Sentinel-1": "ESA",
    "Sentinel-2": "ESA",
    "PRISM": "Oregon State University PRISM Climate Group",
    "AERONET": "NASA",
    "AVISO": "CNES/AVISO",
    "GEBCO": "GEBCO",
    "OpenStreetMap": "OpenStreetMap Foundation",
    "Natural Earth": "Natural Earth",
    "NSIDC": "NSIDC",
    "Argo": "Argo Program",
    "FLUXNET": "FLUXNET",
    "FLUXNET2015": "FLUXNET",
    "CAZy": "CAZy Consortium",
    "InterPro": "EMBL-EBI",
    "Phytozome": "JGI",
    "PRIDE": "EMBL-EBI",
    "DAVID": "NIH",
    "The Plant List": "Royal Botanic Gardens, Kew/Missouri Botanical Garden",
    "World Bank": "World Bank",
    "World Bank Open Data": "World Bank",
    "World Development Indicators": "World Bank",
    "Worldwide Governance Indicators": "World Bank",
    "FAO GAUL": "FAO",
    "FAO FRA": "FAO",
    "NCEP/DOE Reanalysis 2": "NOAA/NCEP and DOE",
    "NCEP CFSR": "NOAA/NCEP",
    "NCEP SST": "NOAA/NCEP",
    "NOAA OI SST V2": "NOAA",
    "NOAA OISST": "NOAA",
    "NOAA ERSST": "NOAA",
    "NOAA Coral Reef Watch": "NOAA",
    "NOAA CoastWatch/OceanWatch": "NOAA",
    "NOAA National Weather Service": "NOAA",
    "NOAA Office for Coastal Management": "NOAA",
    "NOAA NOS": "NOAA",
    "NASA POWER": "NASA",
    "NASA LP DAAC": "NASA/USGS",
    "NASA OBPG": "NASA",
    "NASA MEaSUREs": "NASA",
    "USGS Breeding Bird Survey": "USGS",
    "USGS NAWQA": "USGS",
    "USGS NAS": "USGS",
    "National Land Cover Database": "MRLC/USGS",
    "ECMWF ORAS5": "ECMWF",
    "ECMWF IFS": "ECMWF",
    "GIMMS": "NASA GIMMS",
    "NOAA": "NOAA",
    "NASA": "NASA",
    "IBGE": "IBGE",
    "INMET": "INMET",
    "IPCC": "IPCC",
    "NIST": "NIST",
    "ATCC": "ATCC",
    "ScienceDirect": "Elsevier",
}

# Match modes for the editable alias CSV:
# - exact: cleaned alias must equal the cleaned raw name; generic suffixes are ignored.
# - family_prefix: cleaned raw name may continue after the alias, e.g. GBIF Backbone Taxonomy.
# - domain: raw name or URL contains the listed domain.
CURATED_ALIAS_COLUMNS = [
    "active",
    "canonical_name",
    "entity_type",
    "holder_name",
    "alias",
    "match_mode",
    "notes",
]
CURATED_ALIAS_MATCH_MODES = {"exact", "family_prefix", "domain"}

ENTITY_TYPE_LABEL_MAP = {
    "data_holder_or_provider": "data_holder",
    "data_holder": "data_holder",
    "generic_or_unresolved": "generic_or_ambiguous",
    "generic_or_ambiguous": "generic_or_ambiguous",
    "false_positive_or_empty": "false_positive",
    "false_positive": "false_positive",
}


def normalize_entity_type_label(value):
    text = str(value or "").strip()
    return ENTITY_TYPE_LABEL_MAP.get(text, text)


def normalize_active_flag(value):
    return str(value).strip().casefold() not in {"", "0", "false", "no", "n"}


def normalize_domain_alias(value):
    text = str(value or "").strip().casefold()
    text = re.sub(r"^https?://", "", text)
    text = text.split("/")[0]
    if text.startswith("www."):
        text = text[4:]
    return text


def normalize_domain_match_text(value):
    return re.sub(r"^https?://", "", str(value or "").casefold())


def domain_alias_matches(domain, raw_name):
    domain = normalize_domain_alias(domain)
    if not domain:
        return False
    domain_text = normalize_domain_match_text(raw_name)
    # Match as a domain token, not as an arbitrary substring inside another host.
    pattern = rf"(?<![a-z0-9-])(?:www\.)?{re.escape(domain)}(?=$|[^a-z0-9-])"
    return re.search(pattern, domain_text, flags=re.IGNORECASE) is not None


def load_curated_aliases(path):
    if not path.exists():
        print(f"Curated alias file not found; continuing without alias rules: {path}")
        return pd.DataFrame(columns=CURATED_ALIAS_COLUMNS)

    aliases = pd.read_csv(path).fillna("")
    for column in CURATED_ALIAS_COLUMNS:
        if column not in aliases.columns:
            aliases[column] = ""
    aliases = aliases[CURATED_ALIAS_COLUMNS].copy()
    aliases["active"] = aliases["active"].map(normalize_active_flag)
    aliases["match_mode"] = aliases["match_mode"].astype(str).str.strip().str.casefold()
    aliases.loc[~aliases["match_mode"].isin(CURATED_ALIAS_MATCH_MODES), "match_mode"] = "exact"
    for column in ["canonical_name", "entity_type", "holder_name", "alias", "notes"]:
        aliases[column] = aliases[column].astype(str).str.strip()
    aliases["entity_type"] = aliases["entity_type"].map(normalize_entity_type_label)
    return aliases.loc[aliases["active"]].copy()


def build_curated_alias_indexes(alias_frame):
    exact_map = {}
    prefix_rules = []
    domain_rules = []

    for row in alias_frame.to_dict("records"):
        alias = row.get("alias", "")
        canonical_name = row.get("canonical_name", "")
        if not alias or not canonical_name:
            continue

        record = {
            "canonical_name": canonical_name,
            "entity_type": row.get("entity_type", "") or "data_resource_or_product",
            "holder_name": row.get("holder_name", ""),
            "alias": alias,
            "match_mode": row.get("match_mode", "exact") or "exact",
            "notes": row.get("notes", ""),
        }

        if record["match_mode"] == "domain":
            domain = normalize_domain_alias(alias)
            if domain:
                domain_rules.append((domain, record))
            continue

        alias_key = review_lookup_key(alias)
        if not alias_key:
            continue
        if record["match_mode"] == "family_prefix":
            prefix_rules.append((alias_key, record))
        else:
            exact_map[alias_key] = record

    prefix_rules = sorted(prefix_rules, key=lambda item: len(item[0]), reverse=True)
    domain_rules = sorted(domain_rules, key=lambda item: len(item[0]), reverse=True)
    return exact_map, prefix_rules, domain_rules


def curated_alias_candidate_keys(raw_name):
    key = review_lookup_key(raw_name)
    candidates = []

    def add(candidate):
        candidate = re.sub(r"\s+", " ", str(candidate or "")).strip()
        if candidate and candidate not in candidates:
            candidates.append(candidate)

    add(key)
    stripped = key
    for _ in range(4):
        updated = re.sub(r"^(?:the|a|an)\s+", "", stripped)
        updated = re.sub(
            r"^(?:\d{4}|v\d+(?:\.\d+)*|version\s+\d+(?:\.\d+)*|collection\s+\d+)\s+",
            "",
            updated,
        )
        if updated == stripped:
            break
        add(updated)
        stripped = updated
    return candidates


def product_code_alias_match(raw_name):
    token_text = raw_review_lookup_key(raw_name)
    for prefix_key, record in CURATED_ALIAS_PREFIX_RULES:
        if not re.fullmatch(r"(?:mod|myd|mcd)\d+[a-z0-9]*", prefix_key):
            continue
        if re.search(rf"\b{re.escape(prefix_key)}\b", token_text):
            return record
    return None


def curated_alias_match(raw_name):
    keys = curated_alias_candidate_keys(raw_name)
    for key in keys:
        if key in CURATED_ALIAS_EXACT_MAP:
            return CURATED_ALIAS_EXACT_MAP[key]

    for key in keys:
        for prefix_key, record in CURATED_ALIAS_PREFIX_RULES:
            if key == prefix_key or key.startswith(f"{prefix_key} "):
                return record

    product_code_match = product_code_alias_match(raw_name)
    if product_code_match:
        return product_code_match

    for domain, record in CURATED_ALIAS_DOMAIN_RULES:
        if domain_alias_matches(domain, raw_name):
            return record

    return None


def curated_alias_lookup_by_key(key):
    if key in CURATED_ALIAS_EXACT_MAP:
        return CURATED_ALIAS_EXACT_MAP[key]
    for prefix_key, record in CURATED_ALIAS_PREFIX_RULES:
        if key == prefix_key:
            return record
    return None


PARENTHETICAL_ACRONYM_RE = re.compile(r"\(([A-Z][A-Z0-9&/-]{1,12})\)")
HOLDER_ACRONYM_LEADING_RE = re.compile(
    r"\b(?:agency|administration|authority|bureau|centers?|centres?|commission|council|department|foundation|institute|institution|laborator(?:y|ies)|ministry|organizations?|organisations?|service|survey|university)\b",
    flags=re.IGNORECASE,
)
HOLDER_ACRONYM_ALLOWED_TRAILING_RE = re.compile(
    r"^(?:data|datasets?|data sets?|database|databases|archives?|portals?|websites?|web sites?)$",
    flags=re.IGNORECASE,
)


def parenthetical_acronym_alias_match(raw_name):
    text = str(raw_name or "")
    for match in PARENTHETICAL_ACRONYM_RE.finditer(text):
        record = curated_alias_lookup_by_key(review_lookup_key(match.group(1)))
        if not record:
            continue
        leading_text = text[:match.start()].strip(" .,:;-/")
        trailing_text = text[match.end():].strip(" .,:;-/")
        if record.get("entity_type") == "data_holder":
            if trailing_text and not HOLDER_ACRONYM_ALLOWED_TRAILING_RE.match(trailing_text):
                continue
            if not HOLDER_ACRONYM_LEADING_RE.search(leading_text):
                continue
        return record
    return None


curated_aliases = load_curated_aliases(CURATED_ALIAS_FILE)
CURATED_ALIAS_EXACT_MAP, CURATED_ALIAS_PREFIX_RULES, CURATED_ALIAS_DOMAIN_RULES = build_curated_alias_indexes(curated_aliases)
for row in curated_aliases.to_dict("records"):
    canonical_name = row.get("canonical_name", "")
    holder_name = row.get("holder_name", "")
    if canonical_name and holder_name:
        RESOURCE_TO_HOLDER.setdefault(canonical_name, holder_name)

KNOWN_HOLDER_CANONICAL_BY_KEY = {}
for holder_key in HOLDER_EXACT_KEYS:
    holder_name = SUGGESTED_CANONICAL_MAP.get(holder_key) or holder_key
    KNOWN_HOLDER_CANONICAL_BY_KEY[review_lookup_key(holder_key)] = holder_name
    KNOWN_HOLDER_CANONICAL_BY_KEY[raw_review_lookup_key(holder_key)] = holder_name
for row in curated_aliases.to_dict("records"):
    if row.get("entity_type", "") != "data_holder":
        continue
    canonical_name = row.get("canonical_name", "") or row.get("holder_name", "")
    for value in [row.get("alias", ""), row.get("canonical_name", ""), row.get("holder_name", "")]:
        if value and canonical_name:
            KNOWN_HOLDER_CANONICAL_BY_KEY[review_lookup_key(value)] = canonical_name
            KNOWN_HOLDER_CANONICAL_BY_KEY[raw_review_lookup_key(value)] = canonical_name
print(f"Loaded active curated alias rules: {len(curated_aliases):,} from {CURATED_ALIAS_FILE.name}")


YEAR_OR_SYMBOL_RE = re.compile(r"^(?:\(?#?[-+]?\d+(?:\.\d+)?\)?|\d{4}(?:\s*[-,;/]\s*\d{4})*|[()#.,;:/\-\s]+)$")
BRACKETED_NUMERIC_REFERENCE_RE = re.compile(
    r"^(?=.*\d)(?:[\[\]\d,;/\-\s]|\bor\b)+$"
)
LEADING_BRACKETED_NUMERIC_CITATION_RE = re.compile(
    r"^\[\s*\d+(?:\s*[-,;/]\s*\d+)*\s*\]\s+\S",
    flags=re.IGNORECASE,
)
CITATION_RE = re.compile(
    r"\b(et al\.?|[A-Z][A-Za-z'`-]+\s*(?:and|&)\s*[A-Z][A-Za-z'`-]+\s*\(?[12][0-9]{3}|[A-Z][A-Za-z'`-]+\s+et\s+al\.?\s*\(?[12][0-9]{3}|\([12][0-9]{3}[a-z]?\)|,\s*[12][0-9]{3}[a-z]?\s*$)",
    flags=re.IGNORECASE,
)
LOCAL_RE = re.compile(r"\b(project'?s|this study|our data|own data|local data|field data|personal communication|unpublished)\b", flags=re.IGNORECASE)
GENERIC_RE = re.compile(r"\b(published sequences?|reference sequences?|supplementary data|supporting information|raw data|database|data base|dataset|data set)\b", flags=re.IGNORECASE)
SOFTWARE_RE = re.compile(r"\b(software|package|toolbox)\b", flags=re.IGNORECASE)
SOFTWARE_PATTERN_RE = re.compile(
    r"^(?!.*\b(?:imagery|basemap|base map|world imagery)\b)(?:arcgis|arcmap)(?:\b|\d)|\bgithub\b|github\.com|\b(?:primer[- ]?blast|blast\+|blastn|blastp|blastx|tblastn|ncbi\s+blast|basic local alignment search tool)\b|\b(?:orf\s*finder|ncbi\s+orf\s*finder|pgaap|prokaryotic genomes automatic annotation pipeline)\b|^calib\b|\b(?:sentinel application platform|esa snap|\bsnap\b)\b|\b(?:artificial intelligence for environment(?:\s+(?:and|&)\s+sustainability)?|aries\s+mapping\s+tool|esd[-\s]?aries)\b|^hysplit\b|^qgis\b|^estimates\b|^rast\b|^picrust\b|^genepop\b|^megan\b|^mega(?:\b|\d|\s+(?:x|\d))|^genalex(?:\b|\s*\d)|^spss(?:\b|\s*\d)|^ibm\s+spss\b|^(?:nih\s+)?imagej(?:\b|\d|[-\s])|^fiji\s*/\s*imagej\b|^structure(?:\b|\s*\d)|^structure\s+harvester\b|^clustal\s+omega\b|^arlequin\b|^diva[-\s]?gis\b|^geneious\b|^smart\b(?!\s+database)",
    flags=re.IGNORECASE,
)
SOFTWARE_CANONICAL_PATTERNS = [
    (re.compile(r"^(?!.*\b(?:imagery|basemap|base map|world imagery)\b)(?:esri\s+)?arc\s*gis(?:\b|\s*v?\.?\s*\d|\d)|^(?!.*\b(?:imagery|basemap|base map|world imagery)\b)arcgis(?:\b|\s*v?\.?\s*\d|\d)", flags=re.IGNORECASE), "ArcGIS"),
    (re.compile(r"^arc\s*map\b|^arcmap\b", flags=re.IGNORECASE), "ArcMap"),
    (re.compile(r"^qgis\b", flags=re.IGNORECASE), "QGIS"),
    (re.compile(r"^(?:ibm\s+)?spss\b", flags=re.IGNORECASE), "SPSS"),
    (re.compile(r"^statistica(?:\b|\s*\d)", flags=re.IGNORECASE), "STATISTICA"),
    (re.compile(r"^(?:nih\s+)?imagej(?:\b|\d|[-\s])|^fiji\s*/\s*imagej\b", flags=re.IGNORECASE), "ImageJ"),
    (re.compile(r"^mega(?:\b|\d|\s+(?:x|\d))", flags=re.IGNORECASE), "MEGA"),
    (re.compile(r"^genalex(?:\b|\s*\d)", flags=re.IGNORECASE), "GenAlEx"),
    (re.compile(r"^structure(?:\b|\s*\d)|^structure\s+harvester\b", flags=re.IGNORECASE), "STRUCTURE"),
    (re.compile(r"^diva[-\s]?gis\b", flags=re.IGNORECASE), "DIVA-GIS"),
    (re.compile(r"^geneious\b", flags=re.IGNORECASE), "Geneious"),
    (re.compile(r"\b(?:NCBI\s+)?Primer[- ]?BLAST\b", flags=re.IGNORECASE), "Primer-BLAST"),
    (re.compile(r"\bBasic Local Alignment Search Tool\b|\b(?:NCBI\b.*\bBLAST\b|(?:NCBI[-\s]?)?BLAST(?:n|p|x)\b|Mega[-\s]?BLAST\b|\bBLAST\s+(?:database|server|search|program)\b)", flags=re.IGNORECASE), "BLAST"),
    (re.compile(r"\b(?:NCBI\s+)?ORF\s*Finder\b", flags=re.IGNORECASE), "NCBI ORF Finder"),
    (re.compile(r"\b(?:NCBI\s+)?(?:PGAAP|PGAP|Prokaryotic Genome Annotation Pipeline)\b", flags=re.IGNORECASE), "NCBI PGAP"),
    (re.compile(r"\bHYSPLIT\b", flags=re.IGNORECASE), "HYSPLIT"),
    (re.compile(r"^calib\b", flags=re.IGNORECASE), "Calib"),
    (re.compile(r"^interpro\s*scan\b|^interproscan\b", flags=re.IGNORECASE), "InterProScan"),
    (re.compile(r"\bOcean\s+Data\s+View\b|^ODV\b", flags=re.IGNORECASE), "Ocean Data View"),
    (re.compile(r"\bDigital\s+Shoreline\s+Analysis\s+System\b|\bDSAS\b", flags=re.IGNORECASE), "Digital Shoreline Analysis System"),
]


def canonical_software_match(raw_name):
    text = str(raw_name or "").strip()
    for pattern, canonical_name in SOFTWARE_CANONICAL_PATTERNS:
        if pattern.search(text):
            return canonical_name
    return ""

HOLDER_RE = re.compile(
    r"\b(agency|administration|bureau|center|centre|department|ministry|institute|institution|university|office|commission|authority|laboratory|organisation|organization|council|secretariat)\b",
    flags=re.IGNORECASE,
)
TABLE_OR_FIGURE_RE = re.compile(
    r"^(?:supplementary\s+)?(?:table|tab\.?|figure|fig\.?)\s*[A-Z]?\d+[A-Za-z0-9.\-]*$|^(?:table|figure)\s*S\d+[A-Za-z0-9.\-]*$",
    flags=re.IGNORECASE,
)
SUPPLEMENTARY_LOCAL_RE = re.compile(
    r"\b(?:supplementary|supplemental|supporting|appendix)\s+(?:file|files|material|materials|information|data|dataset|datasets|table|tables|figure|figures|database)\b|^(?:additional|supplementary|supporting)\s+file\s*\d+\b|^(?:data|data\s*set|dataset|database|table|figure|fig\.?|file|appendix)\s*S\d+\b|^S\d+\s*(?:data|data\s*set|dataset|database|table|figure|fig\.?|file|appendix)\b|^(?:electronic\s+)?supplements?\s*\d+[A-Za-z]?\b|^supplements?$",
    flags=re.IGNORECASE,
)
GENERIC_NOISE_FAMILY_RE = re.compile(
    r"^(?:the\s+)?(?:(?:historical|historic|digital|archival|orthorectified|digitized)\s+)?aerial photographs?$"
    r"|^(?:the\s+)?remote sensing(?:\s+(?:data|datasets?|images?|imagery|products?))?$"
    r"|^(?:the\s+)?(?:\d+\s*:\s*[\d,]+\s+)?(?:(?:digital|historical|historic)\s+)?topographic maps?$"
    r"|^(?:the\s+)?(?:national\s+|local\s+|regional\s+|municipal\s+|provincial\s+|county\s+|city\s+|\d{4}\s+)?census data$"
    r"|^(?:the\s+)?(?:\d+\s+)?(?:compiled data from\s+)?previous studies(?:\s*\[[\d,;/\-\s]+\])?$"
    r"|^(?:the\s+)?(?:\d+\s+)?previously published\s+(?:stud(?:y|ies)|papers?|sequences?|sequence data|transcriptomes)$"
    r"|^(?:the\s+)?land cover maps?$"
    r"|^(?:the\s+)?(?:(?:local|regional|municipal|provincial|county|city)\s+)?statistical yearbooks?$"
    r"|^(?:the\s+)?(?:field|household|questionnaire|interview)\s+survey data$",
    flags=re.IGNORECASE,
)
LOCAL_PRIMARY_DATA_RE = re.compile(
    r"\b(?:interviews?|focus\s+groups?|questionnaires?)\b",
    flags=re.IGNORECASE,
)
GENERIC_PHRASE_RE = re.compile(
    r"^(?:the\s+)?(?:published|public|publicly available|existing|online|available|previously published|peer-reviewed|scientific|grey|primary|open)\s+(?:literature|literatures|literature\s+(?:sources?|values?|data)|sources|data|datasets?|data sets?|databases?|repositories|sequences|sequence databases|genomes)(?:\s*,?\s*\[[\d,;/\-\s]+\])?$|^(?:the\s+)?(?:public|publicly available)\s+(?:data sources?|sequence databases?|repositories)(?:\s*,?\s*\[[\d,;/\-\s]+\])?$|^(?:the\s+)?(?:literature|literature data|literature sources|literature review|survey data)(?:\s*,?\s*\[[\d,;/\-\s]+\])?$",
    flags=re.IGNORECASE,
)
GENERIC_EXACT_KEYS = {
    "published sequences",
    "reference sequences",
    "supplementary data",
    "supporting information",
    "raw data",
    "data",
    "dataset",
    "data set",
    "database",
    "data base",
    "databases",
    "public database",
    "public databases",
    "publicly available data",
    "data sources",
    "literature",
    "published data",
    "existing data",
    "online databases",
    "dem",
    "digital elevation model",
    "digital elevation models",
    "digital elevation model dem",
}
ACCESSION_PARENT_PATTERNS = [
    (re.compile(r"\b[AE]-MTAB-\d+\b", flags=re.IGNORECASE), "ArrayExpress", "ArrayExpress accession"),
    (re.compile(r"\bGCA_\d+(?:\.\d+)?\b", flags=re.IGNORECASE), "NCBI Assembly", "GenBank assembly accession"),
    (re.compile(r"\bGCF_\d+(?:\.\d+)?\b", flags=re.IGNORECASE), "NCBI Assembly", "RefSeq assembly accession"),
    (re.compile(r"\b(?:GSE|GSM|GPL)\d{3,}\b", flags=re.IGNORECASE), "NCBI GEO", "GEO accession"),
    (re.compile(r"\b(?:SRP|SRX|SRR|SRS|SRA)\d{3,}\b", flags=re.IGNORECASE), "NCBI SRA", "SRA accession"),
    (re.compile(r"\bPRJNA\d+\b", flags=re.IGNORECASE), "NCBI BioProject", "NCBI BioProject accession"),
    (re.compile(r"\b(?:PRJEB|ERP|ERX|ERR|ERS)\d+\b", flags=re.IGNORECASE), "European Nucleotide Archive", "ENA accession"),
    (re.compile(r"\b(?:PRJDB|DRA|DRX|DRR|DRS)\d+\b", flags=re.IGNORECASE), "DDBJ", "DDBJ accession"),
    (re.compile(r"\b(?:SAMN|SAMD)\d+\b", flags=re.IGNORECASE), "NCBI BioSample", "NCBI BioSample accession"),
    (re.compile(r"\bSAMEA\d+\b", flags=re.IGNORECASE), "European Nucleotide Archive", "European BioSample accession"),
    (re.compile(r"\bphs\d{5,}(?:\.v\d+\.p\d+)?\b", flags=re.IGNORECASE), "dbGaP", "dbGaP accession"),
    (re.compile(r"\b(?:NC|NM|NP|NR|XM|XP|XR)_\d+(?:\.\d+)?\b|\bNZ_[A-Z]{2,}\d+(?:\.\d+)?\b", flags=re.IGNORECASE), "NCBI RefSeq", "RefSeq accession"),
    (re.compile(r"\bCP\d{5,}(?:\.\d+)?\b|^(?:[A-Z]{1,2}\d{5,8}|[A-Z]{4}\d{8,})(?:\.\d+)?$", flags=re.IGNORECASE), "GenBank", "GenBank-style sequence accession"),
]


def accession_parent_resource(raw_name):
    text = str(raw_name or "").strip()
    for pattern, canonical_name, reason in ACCESSION_PARENT_PATTERNS:
        if pattern.search(text):
            return canonical_name, reason
    return "", ""


AUTHOR_WORD_RE = r"(?:[A-ZÀ-ÖØ-Þ][A-Za-zÀ-ÖØ-öø-ÿ'`-]+|(?:de|del|da|do|dos|van|von|le|la))"
AUTHOR_PHRASE_RE = AUTHOR_WORD_RE + r"(?:\s+" + AUTHOR_WORD_RE + r"){0,3}"
REFERENCE_ET_AL_RE = re.compile(
    r"^(" + AUTHOR_PHRASE_RE + r")\s+et\s+al\.?\s*[,;]?\s*[\[(]?([12][0-9]{3}[a-z]?)[\])]?[.:;]?$",
    flags=re.IGNORECASE,
)
REFERENCE_TWO_AUTHOR_RE = re.compile(
    r"^(" + AUTHOR_PHRASE_RE + r")\s*(?:and|&)\s*(" + AUTHOR_PHRASE_RE + r")\s*[,;]?\s*[\[(]?([12][0-9]{3}[a-z]?)[\])]?[.:;]?$",
    flags=re.IGNORECASE,
)
REFERENCE_THREE_PLUS_RE = re.compile(
    r"^(" + AUTHOR_PHRASE_RE + r")\s*,\s+" + AUTHOR_PHRASE_RE + r".*(?:and|&)\s*" + AUTHOR_PHRASE_RE + r"\s*[,;]?\s*[\[(]?([12][0-9]{3}[a-z]?)[\])]?[.:;]?$",
    flags=re.IGNORECASE,
)
REFERENCE_COMMA_YEAR_RE = re.compile(
    r"^([A-Za-zÀ-ÖØ-öø-ÿ0-9][A-Za-zÀ-ÖØ-öø-ÿ0-9'`./& -]{1,90}?)[,;]\s*[\[(]?([12][0-9]{3}[a-z]?)[\])]?[.:;]?$",
    flags=re.IGNORECASE,
)
REFERENCE_INSTITUTION_SPACE_YEAR_RE = re.compile(
    r"^(FAO|IPCC|IBGE|EMBRAPA|INMET|IUCN|USEPA|USDA|WRB|APHA|OECD|ISO|R Core Team|Soil Survey Staff|IUSS Working Group WRB|Census of India)\s+[\[(]?([12][0-9]{3}[a-z]?)[\])]?[.:;]?$",
    flags=re.IGNORECASE,
)
TARGET_REFERENCE_PATTERNS = [
    (re.compile(r"\bTroudet\b.*\b2017\b", flags=re.IGNORECASE), "Troudet et al. (2017)"),
    (re.compile(r"\bTan\b.*\b2017\b", flags=re.IGNORECASE), "Tan et al. (2017)"),
    (re.compile(r"\bStrokal\b.*\b2016\b", flags=re.IGNORECASE), "Strokal et al. (2016)"),
    (re.compile(r"\bSpalding\b.*\b2012\b", flags=re.IGNORECASE), "Spalding et al. (2012)"),
    (re.compile(r"\bSpalding\b.*\b2007\b", flags=re.IGNORECASE), "Spalding et al. (2007)"),
    (re.compile(r"\bSkowno\b.*\bMonyeki\b.*\b2021\b", flags=re.IGNORECASE), "Skowno & Monyeki (2021)"),
    (re.compile(r"\bRowland\b.*\b2020\b", flags=re.IGNORECASE), "Rowland et al. (2020)"),
    (re.compile(r"\bRowland\b.*\b2018\b", flags=re.IGNORECASE), "Rowland et al. (2018)"),
    (re.compile(r"\bPayet\b.*\b2013\b", flags=re.IGNORECASE), "Payet et al. (2013)"),
    (re.compile(r"\bObura\b.*\b2022\b", flags=re.IGNORECASE), "Obura et al. (2022)"),
    (re.compile(r"\bNicholson\b.*\b2024\b", flags=re.IGNORECASE), "Nicholson et al. (2024)"),
    (re.compile(r"\bMurray\b.*\b2020\b", flags=re.IGNORECASE), "Murray et al. (2020)"),
    (re.compile(r"\bMurray\b.*\b2018\b", flags=re.IGNORECASE), "Murray et al. (2018)"),
    (re.compile(r"\bMurray\b.*\b2017\b", flags=re.IGNORECASE), "Murray et al. (2017)"),
    (re.compile(r"\bMarshall\b.*\b2018\b", flags=re.IGNORECASE), "Marshall et al. (2018)"),
    (re.compile(r"\bHoskins\b.*\b2020\b", flags=re.IGNORECASE), "Hoskins et al. (2020)"),
    (re.compile(r"\bGibbs\b.*\bSalmon\b.*\b2015\b", flags=re.IGNORECASE), "Gibbs & Salmon (2015)"),
    (re.compile(r"\bGhoraba\b.*\b2019\b", flags=re.IGNORECASE), "Ghoraba et al. (2019)"),
    (re.compile(r"\bGann\b.*\b2022\b", flags=re.IGNORECASE), "Gann et al. (2022)"),
    (re.compile(r"\bFerrer[- ]Paris\b.*\b2019\b", flags=re.IGNORECASE), "Ferrer-Paris et al. (2019)"),
    (re.compile(r"\bEtter\b.*\b2020\b", flags=re.IGNORECASE), "Etter et al. (2020)"),
    (re.compile(r"\bDonaldson\b.*\b2016\b", flags=re.IGNORECASE), "Donaldson et al. (2016)"),
    (re.compile(r"\bDixon\b.*\b2016\b", flags=re.IGNORECASE), "Dixon et al. (2016)"),
    (re.compile(r"\bDinerstein\b.*\b2017\b", flags=re.IGNORECASE), "Dinerstein et al. (2017)"),
    (re.compile(r"\bDarrah\b.*\b2019\b", flags=re.IGNORECASE), "Darrah et al. (2019)"),
    (re.compile(r"\bBurns\b.*\b2015\b", flags=re.IGNORECASE), "Burns et al. (2015)"),
    (re.compile(r"\bBrooks\b.*\b2016\b", flags=re.IGNORECASE), "Brooks et al. (2016)"),
    (re.compile(r"\bBotts\b.*\b2020\b", flags=re.IGNORECASE), "Botts et al. (2020)"),
    (re.compile(r"\bAbell\b.*\b2008\b", flags=re.IGNORECASE), "Abell et al. (2008)"),
]


def normalize_author_surname(name):
    text = str(name or "").strip(" .,;:()[]")
    if not text:
        return ""
    if text.casefold().startswith(("de ", "del ", "da ", "do ", "dos ", "van ", "von ", "le ", "la ")):
        return text
    if text.isupper() and len(text) <= 8:
        return text
    if text.isupper():
        return text.title()
    return text[:1].upper() + text[1:]


def normalize_reference_author_phrase(name):
    text = str(name or "").strip(" .,;:()[]")
    if not text:
        return ""
    if text.isupper() and len(text) <= 8:
        return text
    if text.isupper():
        return text.title()
    return text[:1].upper() + text[1:]


def canonicalize_reference_name(raw_name):
    text = cleanup_database_name(raw_name)
    text = re.sub(r"\s+", " ", text).strip()
    for pattern, canonical_name in TARGET_REFERENCE_PATTERNS:
        if pattern.search(text):
            return canonical_name

    match = REFERENCE_ET_AL_RE.match(text)
    if match:
        return f"{normalize_author_surname(match.group(1))} et al. ({match.group(2)})"

    match = REFERENCE_TWO_AUTHOR_RE.match(text)
    if match:
        return f"{normalize_author_surname(match.group(1))} & {normalize_author_surname(match.group(2))} ({match.group(3)})"

    match = REFERENCE_THREE_PLUS_RE.match(text)
    if match:
        return f"{normalize_author_surname(match.group(1))} et al. ({match.group(2)})"

    match = REFERENCE_COMMA_YEAR_RE.match(text)
    if match:
        return f"{normalize_reference_author_phrase(match.group(1))} ({match.group(2)})"

    match = REFERENCE_INSTITUTION_SPACE_YEAR_RE.match(text)
    if match:
        return f"{normalize_reference_author_phrase(match.group(1))} ({match.group(2)})"

    return ""


AMBIGUOUS_SHORT_AUTHOR_ALIAS_KEYS = {
    "bold",
    "grace",
    "silva",
}
AUTHOR_CITATION_MARKER_RE = re.compile(
    r"\b(?:et\s+al\.?|and|&)\b.*(?:\[[\d,;/\-\s]+\]|\(?[12][0-9]{3}[a-z]?\)?)",
    flags=re.IGNORECASE,
)
RESOURCE_CONTEXT_RE = re.compile(
    r"\b(database|dataset|data set|data|archive|repository|product|version|release|collection|accession|project|group|consortium|portal|server|mapper|pathway|genes|orthology|taxonomy|map|list|catalogue|catalog)\b",
    flags=re.IGNORECASE,
)


def ambiguous_short_alias_reference(raw_name, alias_match):
    if not alias_match:
        return "", ""
    alias_key = review_lookup_key(alias_match.get("alias", ""))
    canonical_key = review_lookup_key(alias_match.get("canonical_name", ""))
    if alias_key not in AMBIGUOUS_SHORT_AUTHOR_ALIAS_KEYS and canonical_key not in AMBIGUOUS_SHORT_AUTHOR_ALIAS_KEYS:
        return "", ""

    text = str(raw_name or "")
    if RESOURCE_CONTEXT_RE.search(text):
        return "", ""

    canonical_reference = canonicalize_reference_name(text)
    if canonical_reference:
        return canonical_reference, "regex.reference.author_year"
    if AUTHOR_CITATION_MARKER_RE.search(text):
        return "", "regex.reference.author_year_unparsed"
    return "", ""


HOLDER_YEAR_SUFFIX_RE = re.compile(
    r"^(?P<base>.+?)\s*(?:\((?:19|20)\d{2}[a-z]?\)|(?:19|20)\d{2}[a-z]?|,\s*(?:19|20)\d{2}[a-z]?)(?:\s*[.;:])?$",
    flags=re.IGNORECASE,
)
HOLDER_YEAR_RANGE_SUFFIX_RE = re.compile(
    r"^(?P<base>.+?)\s*(?:\((?:19|20)\d{2}\s*[-–]\s*(?:19|20)?\d{2}\)|(?:19|20)\d{2}\s*[-–]\s*(?:19|20)?\d{2})(?:\s*[.;:])?$",
    flags=re.IGNORECASE,
)
URL_OR_DOMAIN_TEXT_RE = re.compile(r"https?://|\b[a-z0-9.-]+\.[a-z]{2,}/", flags=re.IGNORECASE)
SLASH_COMPOSITE_RE = re.compile(r"\b[A-Za-z][A-Za-z0-9 .&'()\-]{0,50}/[A-Za-z0-9][A-Za-z0-9 .&'()\-]{0,50}\b")


def holder_year_reference_match(raw_name):
    text = cleanup_database_name(raw_name)
    for pattern in [HOLDER_YEAR_RANGE_SUFFIX_RE, HOLDER_YEAR_SUFFIX_RE]:
        match = pattern.match(text)
        if not match:
            continue
        base = re.sub(r"^(?:the|a|an)\s+", "", match.group("base").strip(" ,;:()[]"), flags=re.IGNORECASE)
        for key in [review_lookup_key(base), raw_review_lookup_key(base)]:
            canonical_name = KNOWN_HOLDER_CANONICAL_BY_KEY.get(key)
            if canonical_name:
                return canonical_name
    return ""


def slash_composite_source(raw_name):
    text = str(raw_name or "").strip()
    if not text or URL_OR_DOMAIN_TEXT_RE.search(text):
        return False
    return bool(SLASH_COMPOSITE_RE.search(text))


def proposal_tuple(standardized_name, entity_type):
    name = "" if standardized_name is None else str(standardized_name).strip()
    return name, entity_type


def known_entity_type(canonical_name, raw_key="", raw_key_raw=""):
    keys = {
        review_lookup_key(canonical_name),
        raw_review_lookup_key(canonical_name),
        raw_key,
        raw_key_raw,
    }
    if keys & BIBLIOGRAPHIC_EXACT_KEYS:
        return "bibliographic_database"
    if keys & HOLDER_EXACT_KEYS:
        return "data_holder"
    return "data_resource_or_product"



def propose_standardization(raw_name, example_reason=""):
    raw_text = str(raw_name or "").strip()
    key = review_lookup_key(raw_text)
    raw_key = raw_review_lookup_key(raw_text)
    reason_text = str(example_reason or "")
    combined = f"{raw_text} {reason_text}"

    if not raw_key or raw_text.upper() == "MANUAL":
        return proposal_tuple("", "false_positive")
    if YEAR_OR_SYMBOL_RE.match(raw_text):
        return proposal_tuple("", "false_positive")
    if BRACKETED_NUMERIC_REFERENCE_RE.match(raw_text):
        return proposal_tuple("", "citation_or_reference")
    if LEADING_BRACKETED_NUMERIC_CITATION_RE.match(raw_text):
        return proposal_tuple("", "citation_or_reference")
    if SUPPLEMENTARY_LOCAL_RE.search(raw_text):
        return proposal_tuple("", "local_or_project_data")
    if TABLE_OR_FIGURE_RE.match(raw_text):
        return proposal_tuple("", "false_positive")
    cleaned_raw_text = cleanup_database_name(raw_text)
    for pattern, canonical_name, entity_type, reason in EARLY_KNOWN_PATTERN_PROPOSALS:
        if pattern.search(cleaned_raw_text) and entity_type != "generic_or_ambiguous":
            return proposal_tuple(canonical_name, entity_type)
    pre_generic_alias_match = curated_alias_match(raw_text)
    if pre_generic_alias_match:
        guarded_reference, guarded_reference_rule = ambiguous_short_alias_reference(raw_text, pre_generic_alias_match)
        if guarded_reference_rule:
            return proposal_tuple(guarded_reference, "citation_or_reference")
        return proposal_tuple(pre_generic_alias_match["canonical_name"], pre_generic_alias_match["entity_type"])
    pre_generic_acronym_alias_match = parenthetical_acronym_alias_match(raw_text)
    if pre_generic_acronym_alias_match:
        return proposal_tuple(pre_generic_acronym_alias_match["canonical_name"], pre_generic_acronym_alias_match["entity_type"])
    if LOCAL_PRIMARY_DATA_RE.search(raw_text):
        return proposal_tuple("", "local_or_project_data")
    if GENERIC_NOISE_FAMILY_RE.match(raw_text):
        return proposal_tuple("", "generic_or_ambiguous")
    if GENERIC_PHRASE_RE.match(raw_text):
        return proposal_tuple("", "generic_or_ambiguous")
    for pattern, repository_name in DOI_REPOSITORY_PATTERNS:
        if pattern.search(raw_text):
            return proposal_tuple(repository_name, "data_resource_or_product")
    if GENERIC_DOI_RE.search(raw_text):
        return proposal_tuple("", "citation_or_reference")
    software_canonical_name = canonical_software_match(raw_text)
    if software_canonical_name:
        return proposal_tuple(software_canonical_name, "software_or_tool")
    if key in SOFTWARE_EXACT_KEYS or raw_key in SOFTWARE_EXACT_KEYS:
        return proposal_tuple(cleanup_database_name(raw_text), "software_or_tool")
    if SOFTWARE_PATTERN_RE.search(raw_text) or SOFTWARE_RE.search(raw_text):
        return proposal_tuple(cleanup_database_name(raw_text), "software_or_tool")
    cleaned_raw_text = cleanup_database_name(raw_text)
    for pattern, canonical_name, entity_type, reason in EARLY_KNOWN_PATTERN_PROPOSALS:
        if pattern.search(cleaned_raw_text):
            return proposal_tuple(canonical_name, entity_type)
    accession_parent, accession_reason = accession_parent_resource(raw_text)
    if accession_parent:
        return proposal_tuple(accession_parent, "data_resource_or_product")
    if slash_composite_source(raw_text):
        return proposal_tuple(cleanup_database_name(raw_text), "data_resource_or_product")
    for pattern, holder_name, reason in TECHNICAL_INFRASTRUCTURE_HOLDER_PATTERNS:
        if pattern.match(cleaned_raw_text):
            return proposal_tuple(holder_name, "data_holder")
    alias_match = curated_alias_match(raw_text)
    if alias_match:
        guarded_reference, guarded_reference_rule = ambiguous_short_alias_reference(raw_text, alias_match)
        if guarded_reference_rule:
            return proposal_tuple(guarded_reference, "citation_or_reference")
        return proposal_tuple(alias_match["canonical_name"], alias_match["entity_type"])
    acronym_alias_match = parenthetical_acronym_alias_match(raw_text)
    if acronym_alias_match:
        return proposal_tuple(acronym_alias_match["canonical_name"], acronym_alias_match["entity_type"])
    holder_year_match = holder_year_reference_match(raw_text)
    if holder_year_match:
        return proposal_tuple(holder_year_match, "data_holder")
    if LOCAL_RE.search(raw_text):
        return proposal_tuple("", "local_or_project_data")
    canonical_reference = canonicalize_reference_name(raw_text)
    if canonical_reference:
        return proposal_tuple(canonical_reference, "citation_or_reference")
    if CITATION_RE.search(raw_text):
        return proposal_tuple("", "citation_or_reference")
    if key in BIBLIOGRAPHIC_EXACT_KEYS or raw_key in BIBLIOGRAPHIC_EXACT_KEYS:
        canonical_name = SUGGESTED_CANONICAL_MAP.get(key) or SUGGESTED_CANONICAL_MAP.get(raw_key) or cleanup_database_name(raw_text)
        return proposal_tuple(canonical_name, "bibliographic_database")
    if key in HOLDER_EXACT_KEYS or raw_key in HOLDER_EXACT_KEYS:
        canonical_name = SUGGESTED_CANONICAL_MAP.get(key) or SUGGESTED_CANONICAL_MAP.get(raw_key) or cleanup_database_name(raw_text)
        return proposal_tuple(canonical_name, "data_holder")
    if key in GENERIC_EXACT_KEYS or raw_key in GENERIC_EXACT_KEYS:
        return proposal_tuple("", "generic_or_ambiguous")

    if key in SUGGESTED_CANONICAL_MAP:
        canonical_name = SUGGESTED_CANONICAL_MAP[key]
        return proposal_tuple(canonical_name, known_entity_type(canonical_name, key, raw_key))
    if raw_key in SUGGESTED_CANONICAL_MAP:
        canonical_name = SUGGESTED_CANONICAL_MAP[raw_key]
        return proposal_tuple(canonical_name, known_entity_type(canonical_name, key, raw_key))

    cleaned = cleanup_database_name(raw_name)
    for pattern, canonical_name, reason in SUGGESTED_CANONICAL_PATTERNS:
        if pattern.match(cleaned):
            return proposal_tuple(canonical_name, known_entity_type(canonical_name, key, raw_key))

    if HOLDER_RE.search(raw_text):
        return proposal_tuple(cleaned, "data_holder")

    return proposal_tuple(cleaned, "data_resource_or_product")


def format_elapsed_time(seconds):
    seconds = max(float(seconds or 0), 0.0)
    minutes, secs = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h {minutes:02d}m {secs:02d}s"
    if minutes:
        return f"{minutes}m {secs:02d}s"
    return f"{secs}s"


def propose_standardizations_with_progress(raw_stats_frame, progress_every=PROGRESS_EVERY):
    total = len(raw_stats_frame)
    if total == 0:
        return pd.DataFrame(columns=["proposed_standardized_name", "proposed_entity_type"])

    progress_every = max(1, int(progress_every or PROGRESS_EVERY))
    progress_every = max(progress_every, total // 100) if total >= 100 else progress_every
    print(
        f"Creating machine proposals for {total:,} unique raw names "
        f"(status every ~{progress_every:,} names)...",
        flush=True,
    )

    started = time.perf_counter()
    proposal_rows = []
    for done, row in enumerate(raw_stats_frame.itertuples(index=False), start=1):
        proposal_rows.append(propose_standardization(row.raw_name, getattr(row, "example_reason_short", "")))
        if done == 1 or done == total or done % progress_every == 0:
            elapsed = time.perf_counter() - started
            rate = done / elapsed if elapsed else 0.0
            remaining = (total - done) / rate if rate else 0.0
            print(
                f"  {done:,}/{total:,} names ({done / total:.1%}) | "
                f"elapsed {format_elapsed_time(elapsed)} | "
                f"{rate:,.1f} names/s | ETA {format_elapsed_time(remaining)}",
                flush=True,
            )

    return pd.DataFrame(
        proposal_rows,
        columns=["proposed_standardized_name", "proposed_entity_type"],
    )


ENTITY_TYPE_TIE_BREAK_PRIORITY = {
    "data_resource_or_product": 0,
    "data_holder": 1,
    "bibliographic_database": 2,
    "citation_or_reference": 3,
    "software_or_tool": 4,
    "local_or_project_data": 5,
    "generic_or_ambiguous": 6,
    "false_positive": 7,
}


def preferred_entity_type_for_standardized_name(standardized_name, observed_entity_types):
    observed = {str(value or "") for value in observed_entity_types if str(value or "").strip()}
    keys = {
        review_lookup_key(standardized_name),
        raw_review_lookup_key(standardized_name),
    }
    reference_like_name = re.search(r"\([12][0-9]{3}[a-z]?\)\s*$", str(standardized_name or ""))
    if "citation_or_reference" in observed and (canonicalize_reference_name(standardized_name) or CITATION_RE.search(str(standardized_name or "")) or reference_like_name):
        return "citation_or_reference"
    if keys & BIBLIOGRAPHIC_EXACT_KEYS and "bibliographic_database" in observed:
        return "bibliographic_database"
    if keys & HOLDER_EXACT_KEYS and "data_holder" in observed:
        return "data_holder"
    if keys & SOFTWARE_EXACT_KEYS and "software_or_tool" in observed:
        return "software_or_tool"
    return ""


def enforce_one_entity_type_per_standardized_name(frame):
    frame = frame.copy()
    frame["proposed_standardized_name"] = frame["proposed_standardized_name"].astype(str).str.strip()
    frame["proposed_entity_type"] = frame["proposed_entity_type"].astype(str).str.strip()
    nonempty_name = frame["proposed_standardized_name"].ne("")
    if not nonempty_name.any():
        return frame

    type_counts = frame.loc[nonempty_name].groupby("proposed_standardized_name")["proposed_entity_type"].nunique()
    conflict_names = type_counts[type_counts.gt(1)].index
    if len(conflict_names) == 0:
        print("Entity-type check: no standardized-name conflicts found.", flush=True)
        return frame

    weights = (
        frame.loc[nonempty_name]
        .assign(document_count=pd.to_numeric(frame.loc[nonempty_name, "document_count"], errors="coerce").fillna(0).astype(int))
        .groupby(["proposed_standardized_name", "proposed_entity_type"], as_index=False)["document_count"]
        .sum()
    )
    weights["_tie_priority"] = weights["proposed_entity_type"].map(ENTITY_TYPE_TIE_BREAK_PRIORITY).fillna(99).astype(int)
    winners = (
        weights.sort_values(
            ["proposed_standardized_name", "document_count", "_tie_priority", "proposed_entity_type"],
            ascending=[True, False, True, True],
        )
        .drop_duplicates("proposed_standardized_name")
        .set_index("proposed_standardized_name")["proposed_entity_type"]
        .to_dict()
    )

    for name in conflict_names:
        preferred_type = preferred_entity_type_for_standardized_name(
            name,
            weights.loc[weights["proposed_standardized_name"].eq(name), "proposed_entity_type"],
        )
        if preferred_type:
            winners[name] = preferred_type

    frame.loc[nonempty_name, "proposed_entity_type"] = frame.loc[nonempty_name, "proposed_standardized_name"].map(winners).fillna(
        frame.loc[nonempty_name, "proposed_entity_type"]
    )

    print(
        f"Entity-type check: resolved {len(conflict_names):,} standardized names with multiple proposed entity types.",
        flush=True,
    )
    for name in list(conflict_names[:10]):
        type_summary = weights.loc[weights["proposed_standardized_name"].eq(name)].sort_values("document_count", ascending=False)
        type_summary_text = ", ".join(
            f"{row.proposed_entity_type}={int(row.document_count):,}"
            for row in type_summary.itertuples(index=False)
        )
        print(f"  {name}: {type_summary_text} -> {winners.get(name, '')}", flush=True)
    return frame


def build_raw_source_rows():
    rows = []
    context_columns = [column for column in ["reason_short", "run_type", "run_name", "prompt_revision"] if column in database_rows.columns]
    row_columns = TEXT_IDENTIFIER_COLUMNS + context_columns
    for row in database_rows.itertuples(index=False):
        base = {column: getattr(row, column) for column in row_columns}
        for source_order, raw_name in enumerate(split_database_names(row.databases), start=1):
            rows.append({**base, "source_order": source_order, "raw_name": raw_name})
    return pd.DataFrame(rows)



# Build one machine proposal per raw name from the full master-derived data.
pipeline_started = time.perf_counter()
print("Building long raw-source table from LLM-labelled rows...", flush=True)
raw_sources_long = build_raw_source_rows()
print(
    f"Built raw-source table with {len(raw_sources_long):,} mentions in "
    f"{format_elapsed_time(time.perf_counter() - pipeline_started)}.",
    flush=True,
)

if raw_sources_long.empty:
    review_proposals = pd.DataFrame()
else:
    aggregation_started = time.perf_counter()
    print("Aggregating mentions to unique raw names...", flush=True)
    raw_stats = (
        raw_sources_long
        .groupby("raw_name", dropna=False)
        .agg(
            document_count=("file_name", "nunique"),
            example_dois=("doi", compact_unique_join) if "doi" in raw_sources_long.columns else ("file_name", compact_unique_join),
            example_reason_short=("reason_short", compact_unique_join) if "reason_short" in raw_sources_long.columns else ("raw_name", compact_unique_join),
        )
        .reset_index()
    )
    print(
        f"Aggregated to {len(raw_stats):,} unique raw names in "
        f"{format_elapsed_time(time.perf_counter() - aggregation_started)}.",
        flush=True,
    )

    proposals = propose_standardizations_with_progress(raw_stats)
    raw_stats = pd.concat([raw_stats, proposals], axis=1)
    raw_stats = enforce_one_entity_type_per_standardized_name(raw_stats)

    raw_stats["proposed_data_holder"] = raw_stats["proposed_standardized_name"].map(RESOURCE_TO_HOLDER).fillna("")

    standardized_name_count_key = raw_stats["proposed_standardized_name"].astype(str).str.strip()
    standardized_name_count_key = standardized_name_count_key.where(
        standardized_name_count_key.ne(""),
        raw_stats["raw_name"].astype(str).str.strip(),
    )
    raw_stats["proposed_standardized_name_document_count"] = (
        raw_stats.assign(_standardized_name_count_key=standardized_name_count_key)
        .groupby("_standardized_name_count_key")["document_count"]
        .transform("sum")
        .fillna(0)
        .astype(int)
    )

    print("Sorting proposal rows by proposed-name frequency, raw-name frequency, then alphabetically...", flush=True)
    standardized_name_sort_key = raw_stats["proposed_standardized_name"].astype(str).str.strip()
    standardized_name_sort_key = standardized_name_sort_key.where(
        standardized_name_sort_key.ne(""),
        raw_stats["raw_name"].astype(str).str.strip(),
    )
    review_proposals = (
        raw_stats.assign(
            _sort_standardized_name=standardized_name_sort_key.str.casefold(),
            _sort_raw_name=raw_stats["raw_name"].astype(str).str.casefold(),
        )
        .sort_values(
            ["proposed_standardized_name_document_count", "document_count", "_sort_standardized_name", "_sort_raw_name"],
            ascending=[False, False, True, True],
            kind="mergesort",
        )
        .drop(columns=["_sort_standardized_name", "_sort_raw_name"])
        .reset_index(drop=True)
    )

review_output_columns = [
    "raw_name",
    "document_count",
    "proposed_standardized_name",
    "proposed_standardized_name_document_count",
    "proposed_entity_type",
    "proposed_data_holder",
    "example_dois",
    "example_reason_short",
    "review_action",
    "reviewed_standardized_name",
    "reviewed_entity_type",
    "reviewed_data_holder",
    "data_holder_type",
    "data_holder_country_ISO2",
    "URL",
    "reviewer",
    "review_note",
]
human_review_columns = review_output_columns[review_output_columns.index("review_action"):]

review_output = review_proposals.copy()
for column in review_output_columns:
    if column not in review_output.columns:
        review_output[column] = ""
for column in human_review_columns:
    review_output[column] = ""

review_output = review_output[review_output_columns]

print(f"Writing review proposal CSV to {REVIEW_FILE}...", flush=True)
write_pipeline_csv(review_output, REVIEW_FILE)

print(f"Writing review proposal Excel file to {REVIEW_EXCEL_FILE}...", flush=True)
write_pipeline_excel(review_output, REVIEW_EXCEL_FILE)

print(f"Saved automated proposal CSV: {REVIEW_FILE}")
print(f"Saved automated proposal Excel file: {REVIEW_EXCEL_FILE}")
print(f"Rows: {len(review_output):,}")
print(f"Columns: {len(review_output.columns):,}")
print(f"Total notebook 7 proposal-export time: {format_elapsed_time(time.perf_counter() - pipeline_started)}")
display(review_output.head(25))



## 4. Final output file

Notebook `5` stops after creating the automated proposal files for human review. After the review workbook has been manually cleaned and saved under `llm_labelled_data/compiled/manually_cleaned/`, notebook `6` applies the accepted/relabelled decisions to the document-level LLM source mentions.


In [ ]:
# The automated proposal files are written in the previous cell.
print(f"Automated proposal CSV: {REVIEW_FILE}")
print(f"Automated proposal Excel file: {REVIEW_EXCEL_FILE}")
print(f"Manual review workbook expected at: {MANUAL_REVIEW_FILE}")
